#Geneformer In-Silico Perturbation (ISP) Pipeline
### Based on Liu et al. — *Evaluating Foundation Models for In-Silico Perturbation*

---

**Pipeline overview:**
1. Install dependencies
2. Load and QC `.h5ad` files
3. Auto-annotate cell types (marker-based + optional CellTypist)
4. Format AnnData for Geneformer (raw counts → Ensembl IDs → tokenization)
5. Generate baseline embeddings (control & target states)
6. Run separation test (Liu et al. Step 6)
7. Select candidate perturbation genes
8. Run in-silico perturbation (down-regulation / deletion / activation)
9. Compute cosine shift scores
10. Build random baseline & statistical testing
11. Rank genes and output results table
12. Biological validation (pathway enrichment)

---
>

## 0. ⚙️ Install Dependencies

In [ ]:
# Core single-cell stack
!pip install -q anndata scanpy scipy pandas numpy matplotlib seaborn

# Geneformer (from HuggingFace / Theodoris lab)
!pip install -q transformers datasets
!pip install -q pyarrow

# loompy for writing loom files (anndata.write_loom is deprecated)
!pip install -q loompy

# Cell type annotation
!pip install -q celltypist

# Gene ID mapping
!pip install -q mygene

# Pathway enrichment
!pip install -q gseapy

# Stats
!pip install -q statsmodels scikit-learn

# Clone Geneformer repo (for tokenizer + ISP utilities)
import os
if not os.path.exists('/content/Geneformer'):
    !git clone https://huggingface.co/ctheodoris/Geneformer /content/Geneformer

# Install Geneformer package (separate step to avoid %cd issues)
!pip install -q -e /content/Geneformer

# Add to path
import sys
if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')

print('✅ All dependencies installed.')

In [ ]:
!pip install "transformers==4.57.1"
!pip install .

In [ ]:
# ensure datasets updated in Colab environment
!pip install gcsfs==2025.3.0
!pip install datasets==3.6.0

## 1.Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')

H5AD_PATHS = [
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE159977.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE185477.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE189600.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE190487.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE192740.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE202379.h5ad'
]

CONDITION_COL  = None
CONTROL_LABEL  = 'Healthy'
TARGET_LABEL   = 'MASH'
CELLTYPE_COL   = None
FOCUS_CELLTYPES = ['CD16- NK cells', 'Tem/Trm cytotoxic T cells', 'CD16+ NK cells', 'MAIT cells',
                   'Tem/Effector helper T cells', 'NK cells', 'Tem/Temra cytotoxic T cells',
                   'Tcm/Naive helper T cells', 'Naive B cells', 'CRTAM+ gamma-delta T cells',
                   'Hepatocytes', 'T cells', 'Endothelial cells', 'Fibroblasts', 'Macrophages',
                   'Cholangiocytes', 'Mono+mono derived cells', 'B cells', 'Plasma cells',
                   'Memory B cells', 'DC2', 'Classical monocytes', 'pDC',
                   'Intestinal macrophages', 'Non-classical monocytes', 'Resident NK',
                   'Circulating NK/NKT', 'Neutrophils']

# ── DONOR / PATIENT COLUMN ─────────────────────────────────────────────
# Set to the obs column containing donor/patient IDs for patient-aware sampling.
# If None, the pipeline will attempt auto-detection from common column names.
# Set to False to disable patient-aware sampling entirely (not recommended).
DONOR_COL = 'patient_id'  # e.g. 'donor', 'patient_id', 'sample_id'

# ISP patient-aware sampling budget (cells per cell-type/condition per ISP run)
MAX_ISP_CELLS_TOTAL     = 500   # max cells fed to ISP per cell-type
MIN_CELLS_PER_DONOR_ISP = 5     # donors with fewer cells are excluded from ISP sampling

# ── SAVED EMBEDDINGS (optional) ────────────────────────────────────────
# If you already ran Step 6 and saved embeddings, point SAVED_EMBEDDINGS_PATH
# to the .parquet or .csv file to skip re-running the EmbExtractor.
# Set to None to always re-extract embeddings.
SAVED_EMBEDDINGS_PATH = None#'/content/drive/MyDrive/scFM perturbation project/geneformer_embs_new.parquet'

GENE_ID_TYPE   = 'symbol'
SPECIES        = 'human'
PERTURB_MODE   = 'down'
N_CANDIDATE_GENES = 200
MIN_CELLS_PER_STATE = 100
OUTPUT_DIR = '/content/geneformer_isp_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Configuration set.')
print(f'   Control        : {CONTROL_LABEL}')
print(f'   Target         : {TARGET_LABEL}')
print(f'   Perturb        : {PERTURB_MODE}-regulation')
print(f'   Donor col      : {DONOR_COL}')
print(f'   Saved embeddings: {SAVED_EMBEDDINGS_PATH}')


In [ ]:
# ── Auto-detect donor column from common column names ──────────────────
_DONOR_CANDIDATES = [
    'donor', 'donor_id', 'patient', 'patient_id', 'sample', 'sample_id',
    'subject', 'subject_id', 'individual', 'participant', 'Donor', 'Patient',
    'SampleID', 'orig.ident', 'batch',
]

def detect_donor_col(adata, override=None):
    """Return the first matching donor column found in adata.obs."""
    if override is not None and override is not False:
        if override in adata.obs.columns:
            return override
        print(f'   ⚠️  DONOR_COL="{override}" not found in obs.')
        return None
    if override is False:
        return None  # explicitly disabled
    for c in _DONOR_CANDIDATES:
        if c in adata.obs.columns:
            print(f'   Auto-detected donor column: "{c}"')
            return c
    print('   ⚠️  No donor column detected — patient-aware sampling will fall back to random.')
    return None

print('✅ Donor detection helper ready.')


## 2.Load & QC AnnData Objects

In [ ]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

def detect_condition_col(adata):
    #candidates = ['condition','disease','group','diagnosis','phenotype','Status','status','Disease','Condition','sample_type','broad_condition','disease_state']
    candidate = ['broad_condition']
    for c in candidate:
        if c in adata.obs.columns:
            print(f'   Auto-detected condition column: "{c}"')
            print(f'   Values: {adata.obs[c].unique().tolist()}')
            return c
    return None

def load_and_qc(path, condition_col_override=None):
    print(f'\n📂 Loading: {path}')
    adata = sc.read_h5ad(path)
    print(f'   Shape  : {adata.shape[0]:,} cells × {adata.shape[1]:,} genes')
    print(f'   obs    : {list(adata.obs.columns)}')
    if adata.var_names.duplicated().any():
        print('   ⚠️  Duplicate gene names detected — making unique.')
        adata.var_names_make_unique()
    if 'counts' not in adata.layers:
        X = adata.X
        if sp.issparse(X):
            vals = X.data[:1000] if X.nnz > 1000 else X.data
        else:
            vals = np.asarray(X).flat[:1000]
        if np.allclose(vals, np.round(vals), atol=1e-3):
            print('   ✅ .X appears to be raw counts — copying to layers["counts"].')
            adata.layers['counts'] = adata.X.copy()
        else:
            print('   ⚠️  .X may be normalized. Checking adata.raw...')
            if adata.raw is not None:
                print('   ✅ Found adata.raw — using as raw counts.')
                raw_adata = adata.raw.to_adata()
                adata.layers['counts'] = raw_adata[:, adata.var_names].X.copy()
            else:
                print('   ❌ Cannot confirm raw counts. Proceeding with .X — verify manually!')
                adata.layers['counts'] = adata.X.copy()
    mito_prefix = 'MT-' if SPECIES == 'human' else 'mt-'
    adata.var['mt'] = adata.var_names.str.startswith(mito_prefix)
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    n_before = adata.n_obs
    adata = adata[adata.obs['n_genes_by_counts'] > 200].copy()
    adata = adata[adata.obs['pct_counts_mt'] < 20].copy()
    sc.pp.filter_genes(adata, min_cells=10)
    n_after = adata.n_obs
    print(f'   QC: {n_before:,} → {n_after:,} cells retained ({n_before - n_after:,} removed)')
    global CONDITION_COL
    if condition_col_override:
        adata.obs['_condition'] = adata.obs[condition_col_override].astype(str)
    elif CONDITION_COL and CONDITION_COL in adata.obs.columns:
        adata.obs['_condition'] = adata.obs[CONDITION_COL].astype(str)
    else:
        detected = detect_condition_col(adata)
        if detected:
            CONDITION_COL = detected
            adata.obs['_condition'] = adata.obs[detected].astype(str)
        else:
            print('   ❌ Could not detect condition column. Set CONDITION_COL manually.')
            adata.obs['_condition'] = 'unknown'
    print(f'   Conditions: {adata.obs["_condition"].value_counts().to_dict()}')
    return adata

adatas = []
for path in H5AD_PATHS:
    if os.path.exists(path):
        adatas.append(load_and_qc(path))
    else:
        print(f'⚠️  File not found: {path}')
if not adatas:
    raise FileNotFoundError('No .h5ad files loaded. Check H5AD_PATHS.')
print(f'\n✅ Loaded {len(adatas)} dataset(s).')

## 3.Cell Type Annotation
Uses **CellTypist** (pretrained on 36 human tissues) if cell type labels are not already present.

In [ ]:
for adata in adatas:
      if adata.obs.index.duplicated().any():
        print(adata)
        print(f"Duplicate cells before: {adata.obs.index.duplicated().sum()}")
        adata = adata[~adata.obs.index.duplicated(keep='first')]
        print(f"Duplicate cells after: {adata.obs.index.duplicated().sum()}")

In [ ]:
# import celltypist
# from celltypist import models
# import scipy.sparse as sp

# MARKER_DICT = {
#     'Hepatocytes'         : ['ALB','APOB','TTR','APOE','CYP3A4','CYP2E1','HAL'],
#     'Cholangiocytes'      : ['KRT7','KRT19','EPCAM','SOX9','CFTR'],
#     'Hepatic_Stellate'    : ['ACTA2','COL1A1','COL1A2','PDGFRB','LUM','DCN'],
#     'Kupffer_Macrophages' : ['CD68','VSIG4','CLEC4F','TIMD4','C1QA','C1QB'],
#     'Monocytes'           : ['LYZ','S100A8','S100A9','CD14','FCGR3A'],
#     'LSEC'                : ['CLEC4M','LYVE1','STAB2','FCN2','OIT3'],
#     'Portal_Fibroblasts'  : ['MFAP4','EMILIN1','FIBIN','THY1'],
#     'T_cells_CD4'         : ['CD3D','CD3E','CD4','IL7R','TCF7'],
#     'T_cells_CD8'         : ['CD3D','CD3E','CD8A','CD8B','GZMK'],
#     'NK_cells'            : ['GNLY','NKG7','KLRD1','NCR1','FCGR3A'],
#     'B_cells'             : ['CD19','MS4A1','CD79A','CD79B','PAX5'],
#     'Plasma_cells'        : ['MZB1','SDC1','IGHG1','IGKC','XBP1'],
#     'pDC'                 : ['LILRA4','CLEC4C','IL3RA','TCF4'],
#     'cDC1'                : ['CLEC9A','XCR1','CADM1','IDO1'],
#     'cDC2'                : ['CD1C','FCER1A','CLEC10A','S100B'],
# }

# def score_marker_celltypes(adata):
#     adata_tmp = adata.copy()
#     sc.pp.normalize_total(adata_tmp, target_sum=1e4)
#     sc.pp.log1p(adata_tmp)
#     scores = {}
#     for ct, markers in MARKER_DICT.items():
#         present = [g for g in markers if g in adata_tmp.var_names]
#         if len(present) < 2: continue
#         sc.tl.score_genes(adata_tmp, present, score_name=f'_score_{ct}')
#         scores[ct] = adata_tmp.obs[f'_score_{ct}'].values
#     if not scores: return None
#     score_df = pd.DataFrame(scores, index=adata.obs_names)
#     return score_df.idxmax(axis=1)

# def annotate_cell_types(adata, use_celltypist=True):
#     global CELLTYPE_COL
#     ct_candidates = ['cell_type','celltype','CellType','cell_ontology_class','Celltype','leiden_celltypes','anno','annotation','cluster_annotation','broad_celltypes']
#     if CELLTYPE_COL and CELLTYPE_COL in adata.obs.columns:
#         adata.obs['_cell_type'] = adata.obs[CELLTYPE_COL].astype(str)
#         print(f'   Using existing cell type column: "{CELLTYPE_COL}"')
#         print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
#         return adata
#     for c in ct_candidates:
#         if c in adata.obs.columns:
#             CELLTYPE_COL = c
#             adata.obs['_cell_type'] = adata.obs[c].astype(str)
#             print(f'   Auto-detected cell type column: "{c}"')
#             print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
#             return adata
#     print('   No cell type labels found — running CellTypist annotation...')
#     adata_ct = adata.copy()
#     sc.pp.normalize_total(adata_ct, target_sum=1e4)
#     sc.pp.log1p(adata_ct)
#     try:
#         model = models.Model.load(model='Immune_All_Low.pkl')
#         predictions = celltypist.annotate(adata_ct, model=model, majority_voting=True)
#         adata.obs['_cell_type'] = predictions.predicted_labels['majority_voting'].values
#         print('   ✅ CellTypist annotation complete (Immune_All_Low).')
#     except Exception as e:
#         print(f'   ⚠️  CellTypist failed ({e}). Falling back to marker scoring.')
#         marker_labels = score_marker_celltypes(adata)
#         if marker_labels is not None:
#             adata.obs['_cell_type'] = marker_labels.values
#             print('   ✅ Marker-based annotation complete.')
#         else:
#             adata.obs['_cell_type'] = 'Unknown'
#             print('   ❌ Annotation failed. All cells labelled "Unknown".')
#     print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
#     return adata

# for i, adata in enumerate(adatas):
#     print(f'\n--- Dataset {i+1} ---')
#     adatas[i] = annotate_cell_types(adata)
# print('\n✅ Cell type annotation complete.')

In [ ]:
import celltypist
from celltypist import models
import scipy.sparse as sp

liver_gses = ['GSE212837', 'GSE189600', 'GSE174748', 'GSE192740', 'GSE185477', 'GSE202379']
immune_gses = ['GSE270488', 'GSE159977', 'GSE190487', 'GSE192740']

def get_celltypist_model_for_file(filename):
    """Return the appropriate CellTypist model name based on GSE in filename."""
    if filename is None:
        return 'Immune_All_Low.pkl'
    for gse in liver_gses:
        if gse in filename:
            return 'Healthy_Human_Liver.pkl'
    for gse in immune_gses:
        if gse in filename:
            return 'Immune_All_Low.pkl'
    return 'Immune_All_Low.pkl'  # default if no GSE matches

def annotate_cell_types(adata, use_celltypist=True, filename=None):
    global CELLTYPE_COL
    ct_candidates = ['cell_type','celltype','CellType','cell_ontology_class','Celltype','leiden_celltypes','anno','annotation','cluster_annotation','broad_celltypes']
    if CELLTYPE_COL and CELLTYPE_COL in adata.obs.columns:
        adata.obs['_cell_type'] = adata.obs[CELLTYPE_COL].astype(str)
        print(f'   Using existing cell type column: "{CELLTYPE_COL}"')
        print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
        return adata
    for c in ct_candidates:
        if c in adata.obs.columns:
            CELLTYPE_COL = c
            adata.obs['_cell_type'] = adata.obs[c].astype(str)
            print(f'   Auto-detected cell type column: "{c}"')
            print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
            return adata
    print('   No cell type labels found — running CellTypist annotation...')
    model_name = get_celltypist_model_for_file(filename)
    adata_ct = adata.copy()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)
    try:
        model = models.Model.load(model=model_name)
        try:
            predictions = celltypist.annotate(adata_ct, model=model, majority_voting=True)
            adata.obs['_cell_type'] = predictions.predicted_labels['majority_voting'].values
        except Exception:
            # majority_voting fails on some datasets due to over-clustering mismatch
            # fall back to per-cell predictions without voting
            predictions = celltypist.annotate(adata_ct, model=model, majority_voting=False)
            adata.obs['_cell_type'] = predictions.predicted_labels['predicted_labels'].values
            print(f'   ⚠️  Majority voting failed, using per-cell predictions instead.')
        print(f'   ✅ CellTypist annotation complete ({model_name}).')
    except Exception as e:
        adata.obs['_cell_type'] = 'Unknown'
        print(f'   ❌ CellTypist failed ({e}). All cells labelled "Unknown".')
    print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
    return adata

for i, adata in enumerate(adatas):
    print(f'\n--- Dataset {i+1} ---')
    adatas[i] = annotate_cell_types(adata, filename=H5AD_PATHS[i])
print('\n✅ Cell type annotation complete.')

## 4.Map Gene Symbols → Ensembl IDs (Geneformer Vocabulary)

In [ ]:
import mygene
mg = mygene.MyGeneInfo()

def map_genes_to_ensembl(adata, species='human'):
    gene_names = adata.var_names.tolist()
    species_str = 'human' if species == 'human' else 'mouse'
    print(f'   Querying MyGene.info for {len(gene_names):,} genes...')
    results = mg.querymany(gene_names, scopes='symbol' if GENE_ID_TYPE=='symbol' else 'ensembl.gene', fields='ensembl.gene', species=species_str, returnall=False, as_dataframe=True, verbose=False)
    def extract_ensembl(row):
        val = row.get('ensembl.gene', np.nan)
        if isinstance(val, list): return val[0]
        return val
    results['ensembl_id'] = results.apply(extract_ensembl, axis=1)
    gene_map = results['ensembl_id'].dropna().to_dict()
    adata.var['ensembl_id'] = adata.var_names.map(gene_map)
    n_mapped = adata.var['ensembl_id'].notna().sum()
    print(f'   Mapped {n_mapped:,}/{len(gene_names):,} genes to Ensembl IDs.')
    adata = adata[:, adata.var['ensembl_id'].notna()].copy()
    print(f'   Final gene count: {adata.n_vars:,}')
    return adata

for i, adata in enumerate(adatas):
    print(f'\n--- Dataset {i+1} ---')
    adatas[i] = map_genes_to_ensembl(adata, species=SPECIES)
print('\n✅ Gene mapping complete.')

## 5.Tokenize Cells for Geneformer

In [ ]:
import sys
# Make sure Geneformer path is at the beginning of sys.path to prioritize its modules
if '/content/Geneformer' in sys.path:
    sys.path.remove('/content/Geneformer')
sys.path.insert(0, '/content/Geneformer')

import os, numpy as np, scipy.sparse as sp, loompy

# Now import TranscriptomeTokenizer after ensuring compatible transformers version
from geneformer import TranscriptomeTokenizer

TOKENIZED_DIR = os.path.join(OUTPUT_DIR, 'tokenized')
os.makedirs(TOKENIZED_DIR, exist_ok=True)

def prepare_for_tokenizer(adata, dataset_name):
    """Writes a Loom file using loompy directly (anndata.write_loom is deprecated)."""
    adata_tok = adata.copy()
    X = adata_tok.layers['counts'].copy()
    if sp.issparse(X): X = X.toarray()
    X = X.astype(np.float32)
    ensembl_ids = adata_tok.var['ensembl_id'].values.astype(str)
    n_counts = X.sum(axis=1).astype(np.float32)
    matrix = X.T  # genes × cells
    row_attrs = {'ensembl_id': ensembl_ids, 'gene_name': ensembl_ids}
    col_attrs = {
        'CellID'     : np.array(adata_tok.obs_names.tolist()),
        'n_counts'   : n_counts,
        '_cell_type' : np.array(adata_tok.obs['_cell_type'].values.tolist()),
        '_condition' : np.array(adata_tok.obs['_condition'].values.tolist()),
        '_assay_type': np.array(adata_tok.obs['assay_type'].values.tolist()),
        '_patient_id': np.array(adata_tok.obs['patient_id'].values.tolist())
    }
    loom_path = os.path.join(TOKENIZED_DIR, f'{dataset_name}.loom')
    loompy.create(loom_path, matrix, row_attrs, col_attrs)
    print(f'   Saved loom: {loom_path}  ({matrix.shape[1]} cells, {matrix.shape[0]} genes)')
    return loom_path

loom_paths = []
for i, adata in enumerate(adatas):
    name = f'dataset_{i+1}'
    print(f'\n--- Preparing dataset {i+1} for tokenizer ---')
    loom_paths.append(prepare_for_tokenizer(adata, name))

print('\n🔤 Running Geneformer tokenizer...')
tk = TranscriptomeTokenizer(custom_attr_name_dict={'_cell_type':'cell_type','_condition':'condition',
                                                   '_assay_type':'assay_type','_patient_id':'patient_id'}, nproc=4)
TOKENIZED_DATA_DIR = os.path.join(OUTPUT_DIR, 'tokenized_data')
os.makedirs(TOKENIZED_DATA_DIR, exist_ok=True)
tk.tokenize_data(data_directory=TOKENIZED_DIR, output_directory=TOKENIZED_DATA_DIR, output_prefix='geneformer_input', file_format='loom')
print('✅ Tokenization complete.')
print(f'   Tokenized dataset saved to: {TOKENIZED_DATA_DIR}')

## 6.Generate Geneformer Cell Embeddings

In [ ]:
import sys
if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')
import os, torch, pandas as pd
from geneformer import EmbExtractor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
EMB_DIR = os.path.join(OUTPUT_DIR, 'embeddings')
os.makedirs(EMB_DIR, exist_ok=True)
DATASET_PATH = os.path.join(TOKENIZED_DATA_DIR, 'geneformer_input.dataset')

# ── Option A: Load pre-computed embeddings from disk ───────────────────
# Set SAVED_EMBEDDINGS_PATH in the Config cell to skip re-extraction.
def load_embeddings_from_file(path):
    """Load embeddings saved as .parquet or .csv.
    Required columns: cell_type, condition  +  numeric emb_* columns.
    """
    print(f'   Loading embeddings from: {path}')
    if path.endswith('.parquet'):
        df = pd.read_parquet(path)
    elif path.endswith('.csv'):
        df = pd.read_csv(path, index_col=0)
    else:
        raise ValueError(f'Unsupported format: {path}  (expected .parquet or .csv)')
    missing = [c for c in ['cell_type', 'condition'] if c not in df.columns]
    if missing:
        raise ValueError(f'Saved embedding file missing columns: {missing}')
    print(f'   Shape: {df.shape}  |  '
          f'Conditions: {df["condition"].unique().tolist()}  |  '
          f'Cell types: {df["cell_type"].nunique()}')
    return df

if SAVED_EMBEDDINGS_PATH and os.path.exists(SAVED_EMBEDDINGS_PATH):
    # ── Load from saved file ─────────────────────────────────────────────
    print('📂 Loading pre-computed embeddings (skipping EmbExtractor)...')
    embeddings = load_embeddings_from_file(SAVED_EMBEDDINGS_PATH)
    print('✅ Embeddings loaded from saved file.')
else:
    if SAVED_EMBEDDINGS_PATH:
        print(f'⚠️  SAVED_EMBEDDINGS_PATH set but file not found: {SAVED_EMBEDDINGS_PATH}')
        print('   Falling back to EmbExtractor...')
    # ── Option B: Extract embeddings with Geneformer EmbExtractor ──────
    print('🤖 Extracting embeddings with Geneformer EmbExtractor...')
    embex = EmbExtractor(
        model_type='Pretrained', num_classes=0, emb_mode='cell',
        cell_emb_style='mean_pool',
        filter_data={'cell_type': FOCUS_CELLTYPES, 'assay_type': 'scRNA-seq'},
        max_ncells=None, emb_layer=-1,
        emb_label=['cell_type', 'condition','patient_id'], nproc=4,
    )
    embeddings = embex.extract_embs(
        model_directory='ctheodoris/Geneformer',
        input_data_file=DATASET_PATH,
        output_directory=EMB_DIR,
        output_prefix='geneformer_embs',
    )
    # Save for future reuse
    emb_save_path = os.path.join(EMB_DIR, 'geneformer_embs.parquet')
    embeddings.to_parquet(emb_save_path, index=True)
    print(f'   Embeddings saved to: {emb_save_path}')
    print(f'   ➡️  Set SAVED_EMBEDDINGS_PATH = "{emb_save_path}" in Config to reuse.')
    print('✅ Embeddings extracted.')

print(f'   Embedding matrix shape: {embeddings.shape}')
print(f'   Columns sample: {embeddings.columns.tolist()[:5]}')


## 7.Separation Test (as per Liu et al.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8,
})
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')


def run_separation_test_publishable(
        embeddings_df, cell_type, control_label, target_label,
        emb_cols=None, donor_col='donor'):
    """
    Separation test with patient-aware statistics for publication.

    Computes:
    ─ Per-cell cosine similarity margin (Liu et al. Step 6)
    ─ Mann-Whitney U test (two-tailed composite p-value)
    ─ Patient-level Cohen's d (guards against pseudoreplication)
    ─ Balanced accuracy from cross-validated logistic classifier

    Returns dict with: passes, p_value, sep_score, cohens_d,
    balanced_accuracy, n_ctrl, n_tgt, n_donors_ctrl, n_donors_tgt,
    ctrl_margins, tgt_margins.
    """
    ct_mask   = embeddings_df['cell_type'] == cell_type
    ctrl_mask = ct_mask & (embeddings_df['condition'] == control_label)
    tgt_mask  = ct_mask & (embeddings_df['condition'] == target_label)
    n_ctrl, n_tgt = ctrl_mask.sum(), tgt_mask.sum()

    empty = {'passes': False, 'p_value': 1.0, 'sep_score': 0.0,
             'cohens_d': None, 'balanced_accuracy': 0.5,
             'n_ctrl': n_ctrl, 'n_tgt': n_tgt,
             'n_donors_ctrl': 0, 'n_donors_tgt': 0,
             'ctrl_margins': np.array([]), 'tgt_margins': np.array([])}

    if n_ctrl < MIN_CELLS_PER_STATE or n_tgt < MIN_CELLS_PER_STATE:
        print(f'   ⚠️  {cell_type}: insufficient cells '
              f'(ctrl={n_ctrl}, tgt={n_tgt}). Skipping.')
        return empty

    if emb_cols is None:
        meta = {'cell_type', 'condition', 'donor', 'cell_id', 'index'}
        emb_cols = [c for c in embeddings_df.columns
                    if c not in meta
                    and pd.api.types.is_numeric_dtype(embeddings_df[c])]

    ctrl_embs = embeddings_df.loc[ctrl_mask, emb_cols].values.astype(np.float32)
    tgt_embs  = embeddings_df.loc[tgt_mask,  emb_cols].values.astype(np.float32)
    ctrl_centroid = ctrl_embs.mean(axis=0, keepdims=True)
    tgt_centroid  = tgt_embs.mean(axis=0,  keepdims=True)

    # Liu et al. cosine similarity margins
    ctrl_margins = (cosine_similarity(ctrl_embs, ctrl_centroid).flatten() -
                    cosine_similarity(ctrl_embs, tgt_centroid).flatten())
    tgt_margins  = (cosine_similarity(tgt_embs,  tgt_centroid).flatten() -
                    cosine_similarity(tgt_embs,  ctrl_centroid).flatten())

    _, p_ctrl = mannwhitneyu(ctrl_margins, np.zeros(len(ctrl_margins)),
                              alternative='greater')
    _, p_tgt  = mannwhitneyu(tgt_margins,  np.zeros(len(tgt_margins)),
                              alternative='greater')
    sep_score  = (ctrl_margins.mean() + tgt_margins.mean()) / 2
    p_combined = max(p_ctrl, p_tgt)

    # ── Patient-level Cohen's d ─────────────────────────────────────────
    # Averages per donor before computing effect size to avoid
    # pseudoreplication from treating cells as independent observations.
    n_donors_ctrl = n_donors_tgt = 0
    cohens_d = None
    actual_donor_col = None
    for dc in ([donor_col] if donor_col else []) + ['donor', 'Donor', 'patient_id']:
        if dc and dc in embeddings_df.columns:
            actual_donor_col = dc
            break

    if actual_donor_col:
        ctrl_donors = embeddings_df.loc[ctrl_mask, actual_donor_col].values
        tgt_donors  = embeddings_df.loc[tgt_mask,  actual_donor_col].values
        n_donors_ctrl = len(np.unique(ctrl_donors))
        n_donors_tgt  = len(np.unique(tgt_donors))
        ctrl_donor_means = (
            pd.Series(ctrl_margins, index=ctrl_donors).groupby(level=0).mean()
        )
        tgt_donor_means  = (
            pd.Series(tgt_margins,  index=tgt_donors).groupby(level=0).mean()
        )
        if len(ctrl_donor_means) >= 2 and len(tgt_donor_means) >= 2:
            pooled_sd = np.sqrt(
                (ctrl_donor_means.std()**2 + tgt_donor_means.std()**2) / 2 + 1e-10
            )
            cohens_d = float(
                (tgt_donor_means.mean() - ctrl_donor_means.mean()) / pooled_sd
            )

    # ── Balanced accuracy (linear classifier, 3-fold CV) ───────────────
    X_all = np.vstack([ctrl_embs, tgt_embs])
    y_all = np.array([0]*len(ctrl_embs) + [1]*len(tgt_embs))
    ba_scores = []
    n_splits  = min(3, min(n_ctrl, n_tgt) // 10)
    if n_splits >= 2:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        for train_idx, test_idx in skf.split(X_all, y_all):
            clf = LogisticRegression(max_iter=500, C=0.1, random_state=42,
                                      solver='lbfgs')
            clf.fit(X_all[train_idx], y_all[train_idx])
            ba_scores.append(balanced_accuracy_score(
                y_all[test_idx], clf.predict(X_all[test_idx])
            ))
    balanced_accuracy = float(np.mean(ba_scores)) if ba_scores else 0.5

    passes = (p_combined < 0.05) and (sep_score > 0)

    return {
        'passes':             passes,
        'p_value':            float(p_combined),
        'sep_score':          float(sep_score),
        'cohens_d':           cohens_d,
        'balanced_accuracy':  balanced_accuracy,
        'n_ctrl':             int(n_ctrl),
        'n_tgt':              int(n_tgt),
        'n_donors_ctrl':      int(n_donors_ctrl),
        'n_donors_tgt':       int(n_donors_tgt),
        'ctrl_margins':       ctrl_margins,
        'tgt_margins':        tgt_margins,
    }


emb_meta_cols    = {'cell_type', 'condition', 'cell_id', 'donor'}
emb_feature_cols = [c for c in embeddings.columns
                    if c not in emb_meta_cols
                    and pd.api.types.is_numeric_dtype(embeddings[c])]

cell_types_to_test = (FOCUS_CELLTYPES if FOCUS_CELLTYPES
                      else embeddings['cell_type'].unique().tolist())
# Only test types present in embeddings
cell_types_to_test = [ct for ct in cell_types_to_test
                       if ct in embeddings['cell_type'].values]

# Detect donor column across all loaded datasets
_donor_col_in_emb = None
for _dc in (['donor', 'Donor', 'donor_id', 'patient_id', 'sample_id']):
    if _dc in embeddings.columns:
        _donor_col_in_emb = _dc
        print(f'   Using donor column in embeddings: "{_donor_col_in_emb}"')
        break

print(f'\n🔬 Running separation tests for {len(cell_types_to_test)} cell type(s)...')
separation_results = {}
for ct in cell_types_to_test:
    res = run_separation_test_publishable(
        embeddings, ct, CONTROL_LABEL, TARGET_LABEL,
        emb_cols=emb_feature_cols,
        donor_col=_donor_col_in_emb
    )
    separation_results[ct] = res

passing_celltypes = [ct for ct, r in separation_results.items() if r['passes']]
print(f'\n✅ {len(passing_celltypes)}/{len(cell_types_to_test)} cell types pass '
      f'separation test.')
print(f'   Proceeding with: {passing_celltypes}')

# Print detailed summary table
sep_summary = pd.DataFrame([
    {
        'cell_type':          ct,
        'passes':             r['passes'],
        'sep_score':          round(r['sep_score'], 5),
        'p_value':            f"{r['p_value']:.2e}",
        'cohens_d':           round(r['cohens_d'], 3) if r['cohens_d'] is not None else 'N/A',
        'balanced_accuracy':  round(r['balanced_accuracy'], 3),
        'n_ctrl':             r['n_ctrl'],
        'n_tgt':              r['n_tgt'],
        'n_donors_ctrl':      r['n_donors_ctrl'],
        'n_donors_tgt':       r['n_donors_tgt'],
    }
    for ct, r in separation_results.items()
])
sep_summary_path = os.path.join(OUTPUT_DIR, 'separation_test_summary.csv')
sep_summary.to_csv(sep_summary_path, index=False)
print('\nSeparation Test Summary:')
display(sep_summary)


In [ ]:
# ── Publishable Separation Test Figures ────────────────────────────────
# Produces two publication-ready figures:
#   1. Per-cell-type panel: PCA scatter (Panel A) + cosine margin violin (Panel B)
#   2. Multi-cell-type summary: separation scores, balanced accuracy, Cohen's d

SEP_FIG_DIR = os.path.join(OUTPUT_DIR, 'separation_test_figures')
os.makedirs(SEP_FIG_DIR, exist_ok=True)

PALETTE = {CONTROL_LABEL: '#4C72B0', TARGET_LABEL: '#DD8452'}
SIG_GREEN = '#2ecc71'
FAIL_RED  = '#e74c3c'

# ── Compute 2D PCA coordinates for visualisation ─────────────────────
pca = PCA(n_components=2, random_state=42)
all_emb_vals = embeddings[emb_feature_cols].values
coords_2d    = pca.fit_transform(all_emb_vals)
emb_plot = embeddings.copy()
emb_plot['PC1'] = coords_2d[:, 0]
emb_plot['PC2'] = coords_2d[:, 1]

# ── Figure 1: Per-cell-type panels ──────────────────────────────────
cell_types_with_data = [ct for ct in passing_celltypes
                         if ct in separation_results]
n_cts = len(cell_types_with_data)

if n_cts > 0:
    fig, axes = plt.subplots(n_cts, 2,
                              figsize=(13, 4.5 * n_cts),
                              squeeze=False)
    fig.suptitle(
        f'Geneformer Cell-State Separation Test\n'
        f'{CONTROL_LABEL} vs {TARGET_LABEL}',
        fontsize=14, fontweight='bold', y=1.01
    )

    for row_idx, ct in enumerate(cell_types_with_data):
        res = separation_results[ct]
        ct_mask = emb_plot['cell_type'] == ct
        ct_df   = emb_plot[ct_mask]

        # Panel A: PCA scatter
        ax = axes[row_idx, 0]
        for cond, color in PALETTE.items():
            m = ct_df['condition'] == cond
            ax.scatter(ct_df.loc[m, 'PC1'], ct_df.loc[m, 'PC2'],
                       c=color, alpha=0.45, s=14, linewidths=0,
                       label=cond, rasterized=True)
        ax.set_xlabel(
            f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=10)
        ax.set_ylabel(
            f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=10)
        ax.set_title(f'{ct}\nPCA Embedding Space',
                      fontsize=10, fontweight='bold')
        ax.legend(fontsize=9, markerscale=1.5, framealpha=0.8)

        d_str = (f"Cohen's d = {res['cohens_d']:.2f}"
                 if res['cohens_d'] is not None else "Cohen's d = N/A")
        status_str = 'PASS ✅' if res['passes'] else 'FAIL ❌'
        ax.text(0.04, 0.97,
                f"Sep. score = {res['sep_score']:.4f}\n"
                f"p = {res['p_value']:.2e}\n"
                f"{d_str}\n"
                f"Bal. acc = {res['balanced_accuracy']:.2f}\n"
                f"[{status_str}]",
                transform=ax.transAxes, fontsize=8, va='top',
                bbox=dict(boxstyle='round,pad=0.4', fc='white',
                          alpha=0.85, ec='#cccccc'))

        # Panel B: Cosine margin violins
        ax2 = axes[row_idx, 1]
        margin_parts = []
        if len(res['ctrl_margins']) > 0:
            margin_parts.append(pd.DataFrame({
                'Cosine Margin': res['ctrl_margins'],
                'Condition': CONTROL_LABEL
            }))
        if len(res['tgt_margins']) > 0:
            margin_parts.append(pd.DataFrame({
                'Cosine Margin': res['tgt_margins'],
                'Condition': TARGET_LABEL
            }))
        if margin_parts:
            margin_df = pd.concat(margin_parts, ignore_index=True)
            sns.violinplot(data=margin_df, x='Condition', y='Cosine Margin',
                           palette=PALETTE, ax=ax2, inner='box',
                           linewidth=1.2, cut=0, saturation=0.85)
            ax2.axhline(0, color='black', lw=1.2, ls='--', alpha=0.6,
                        label='Separation boundary')
            ax2.set_xlabel('', fontsize=10)
            ax2.set_ylabel('Cosine Similarity Margin\n(toward own centroid)',
                           fontsize=9)
            ax2.set_title(f'{ct}\nCell-State Cosine Margins',
                          fontsize=10, fontweight='bold')
            ax2.legend(fontsize=8, loc='lower right')

    plt.tight_layout()
    fig1_path = os.path.join(SEP_FIG_DIR, 'separation_per_celltype.pdf')
    plt.savefig(fig1_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig(fig1_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'   Saved: {fig1_path}')

# ── Figure 2: Multi-cell-type summary ───────────────────────────────
if len(separation_results) >= 2:
    ct_names   = list(separation_results.keys())
    sep_scores = [separation_results[ct]['sep_score']         for ct in ct_names]
    bal_accs   = [separation_results[ct]['balanced_accuracy'] for ct in ct_names]
    cohens_ds  = [(separation_results[ct]['cohens_d'] or 0.0) for ct in ct_names]
    pvals      = [separation_results[ct]['p_value']           for ct in ct_names]
    colors     = [SIG_GREEN if separation_results[ct]['passes'] else FAIL_RED
                  for ct in ct_names]

    fig2, axes2 = plt.subplots(1, 3, figsize=(16, max(4, len(ct_names)*0.5 + 2)))
    fig2.suptitle('Separation Test Summary — All Cell Types',
                   fontsize=13, fontweight='bold')

    # Separation scores
    ax = axes2[0]
    bars = ax.barh(ct_names, sep_scores, color=colors,
                    edgecolor='black', linewidth=0.6)
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('Separation Score', fontsize=11)
    ax.set_title('A. Cosine Separation Score', fontsize=11, fontweight='bold')
    for bar, pv in zip(bars, pvals):
        sig = ('***' if pv < 0.001 else '**' if pv < 0.01
               else '*' if pv < 0.05 else 'ns')
        ax.text(bar.get_width() + max(sep_scores)*0.02,
                bar.get_y() + bar.get_height()/2,
                sig, va='center', fontsize=10)

    # Balanced accuracy
    ax2b = axes2[1]
    ax2b.barh(ct_names, bal_accs, color=colors,
               edgecolor='black', linewidth=0.6)
    ax2b.axvline(0.5, color='gray', lw=1.2, ls='--', alpha=0.7,
                 label='Chance (0.5)')
    ax2b.set_xlim(0, 1.05)
    ax2b.set_xlabel('Balanced Accuracy (3-fold CV)', fontsize=11)
    ax2b.set_title('B. Linear Classifier Accuracy', fontsize=11, fontweight='bold')
    ax2b.legend(fontsize=8)

    # Cohen's d
    ax3 = axes2[2]
    ax3.barh(ct_names, cohens_ds, color=colors,
              edgecolor='black', linewidth=0.6)
    ax3.axvline(0,   color='black', lw=1)
    ax3.axvline(0.5, color='gray', lw=1, ls=':', alpha=0.6, label="d=0.5 (medium)")
    ax3.axvline(0.8, color='gray', lw=1, ls='--', alpha=0.6, label="d=0.8 (large)")
    ax3.set_xlabel("Cohen's d (patient-level)", fontsize=11)
    ax3.set_title('C. Patient-Level Effect Size', fontsize=11, fontweight='bold')
    ax3.legend(fontsize=8)

    legend_handles = [
        mpatches.Patch(color=SIG_GREEN, label='Passes (p<0.05)'),
        mpatches.Patch(color=FAIL_RED,  label='Fails'),
    ]
    fig2.legend(handles=legend_handles, loc='lower center', ncol=2,
                fontsize=10, framealpha=0.9, bbox_to_anchor=(0.5, -0.06))

    plt.tight_layout()
    fig2_path = os.path.join(SEP_FIG_DIR, 'separation_summary.pdf')
    plt.savefig(fig2_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig(fig2_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'   Saved: {fig2_path}')

print('\n✅ Separation test figures saved.')


## 8.Select Candidate Perturbation Genes

In [ ]:
MASH_GENES = [

    # ── GWAS / genetic risk (10) ──────────────────────────────────────────────
    # Validated GWAS loci — expression is context-dependent, not constitutive.
    # Independent of expression datasets — strongest unbiased evidence.
    "PNPLA3",    # rs738409 I148M — largest MASH risk locus; lipid droplet
    "TM6SF2",    # E167K — VLDL secretion defect; steatosis + fibrosis
    "MBOAT7",    # rs641738 — phosphatidylinositol remodelling
    "HSD17B13",  # rs72613567 — loss-of-function PROTECTIVE against MASH
    "GCKR",      # rs1260326 — glucokinase regulator; de novo lipogenesis
    "SAMM50",    # rs3761472 — mitochondrial membrane; fibrosis GWAS
    "NCAN",      # rs2228603 — hepatic steatosis GWAS
    "ABCB4",     # rs2109505 — biliary phosphatidylcholine; fibrosis risk
    "CIDEB",     # rs1805081 — lipid droplet fusion; protective variant
    "PTPN11",    # rs11066301 — protein tyrosine phosphatase; MASH GWAS

    # ── ECM / fibrosis / HSC activation (12) ─────────────────────────────────
    # Near-absent in healthy liver; specifically upregulated in activated HSCs.
    "COL1A1",    # fibrillar collagen — canonical fibrosis hallmark
    "COL1A2",
    "COL3A1",    # reticular collagen
    "ACTA2",     # α-SMA — activated HSC marker; absent in quiescent HSCs
    "LOXL1",     # collagen crosslinking; M5 fibrosis hub (Piras 2024)
    "LOXL2",
    "TIMP1",     # MMP inhibitor — blocks fibrosis resolution
    "TIMP2",
    "MMP2",      # matrix metalloproteinase — ECM remodelling
    "MMP9",
    "PDGFRB",    # PDGF receptor β — HSC proliferation; upregulated on activation
    "SMOC2",     # M5 fibrosis coexpression module driver (Piras & DiStefano 2024)

    # ── TGF-β / SMAD fibrogenic signalling (5) ───────────────────────────────
    "TGFB1",     # master pro-fibrotic cytokine; regulated not constitutive
    "TGFB2",
    "SMAD2",     # canonical fibrogenic effector
    "SMAD3",
    "TGFBR1",    # ALK5 — TGF-β type I receptor

    # ── Innate immune / Kupffer cells / NLRP3 (12) ───────────────────────────
    # All inducible — not constitutively expressed at high levels.
    "TNF",       # TNFα — inducible; central MASH pro-inflammatory cytokine
    "IL6",       # inducible; elevated in MASH
    "IL1B",      # NLRP3 product; specifically upregulated in MASH
    "IL18",      # NLRP3 product; elevated in MASH
    "CXCL10",    # IP-10 — IFN-induced; markedly elevated in MASH
    "CCL2",      # MCP-1 — monocyte recruitment; induced in MASH
    "CCR2",      # MCP-1 receptor on infiltrating monocytes
    "CD68",      # macrophage marker; regulated by activation state
    "TLR4",      # LPS receptor; key innate immune MASH driver
    "MYD88",     # central TLR adaptor; regulated
    "NLRP3",     # inflammasome sensor — key MASH driver; therapeutic target
    "CASP1",     # inflammasome effector caspase; IL-1β/IL-18 maturation

    # ── Lipid-associated macrophage (LAM) markers (5) ────────────────────────
    # MASH-specific macrophage subpopulation (Guilliams 2022, 2025).
    # Specifically elevated in MASH — not expressed in healthy liver macrophages.
    "TREM2",     # LAM hub; strongly elevated in MASH macrophages
    "GPNMB",     # glycoprotein NMB; LAM marker; fibrosis correlate
    "SPP1",      # osteopontin; LAM/scar macrophage; MASH progression marker
    "FABP4",     # lipid-associated macrophage marker; disease-regulated
    "LGALS3",    # galectin-3; macrophage activation; fibrosis biomarker

    # ── Adaptive immunity — T cells / checkpoints (4) ────────────────────────
    "CD8A",      # cytotoxic T cell marker
    "FOXP3",     # Treg master TF; regulated by immune context
    "PDCD1",     # PD-1 — T cell exhaustion; elevated in chronic MASH
    "IFNG",      # IFN-γ — inducible Th1 cytokine; Kupffer cell priming

    # ── Lipid metabolism — specifically dysregulated in MASH (9) ─────────────
    # Regulated by nutritional/hormonal state — not constitutively maximal.
    "FASN",      # fatty acid synthase; upregulated in MASH hepatocytes
    "ACACA",     # ACC1 — rate-limiting DNL; induced by insulin/SREBP
    "SREBF1",    # SREBP-1c — master lipogenic TF; induced in MASH
    "DGAT2",     # diacylglycerol acyltransferase; TG synthesis; regulated
    "PLIN2",     # lipid droplet protein; regulated by lipid load
    "CD36",      # FA translocase; markedly upregulated in MASH hepatocytes
    "ACSL4",     # long-chain acyl-CoA synthetase; ferroptosis sensitiser
    "CPT1A",     # FAO rate-limiting step; suppressed in MASH
    "PPARA",     # PPARα — FAO master regulator; most suppressed NR in MASH

    # ── Nuclear receptors / hepatocyte TFs (8) ───────────────────────────────
    # Expression regulated by nutritional state and disease — not constitutive.
    "PPARG",     # PPARγ — lipogenesis; HSC quiescence
    "NR1H4",     # FXR — bile acid/lipid homeostasis; therapeutic target
    "HNF4A",     # central MASH network hub; reduced in advanced disease
    "CEBPA",     # hepatocyte TF; suppressed in MASH fibrosis
    "FOXO1",     # gluconeogenesis + insulin signalling integrator
    "NR0B2",     # SHP — FXR-inducible metabolic gatekeeper
    "THRB",      # THR-β — target of resmetirom (FDA approved 2024)
    "KLF6",      # Krüppel-like factor 6 — HSC activation TF

    # ── Oxidative stress / antioxidant / ferroptosis (8) ─────────────────────
    # Stress-responsive — induced by ROS, not constitutively maximal.
    "NFE2L2",    # NRF2 — stress-inducible master antioxidant TF
    "HMOX1",     # heme oxygenase-1; strongly stress-inducible
    "GPX4",      # ferroptosis gatekeeper; regulated by lipid peroxides
    "SLC7A11",   # xCT — cystine import; ferroptosis; regulated
    "NOX4",      # NADPH oxidase 4; ROS production; fibrosis driver
    "SIRT1",     # NAD+-deacetylase; MASH-suppressed; not constitutive
    "TXNIP",     # thioredoxin-interacting protein; oxidative stress sensor
    "SOD2",      # mitochondrial SOD; regulated by oxidative stress

    # ── Cell death — apoptosis / necroptosis / pyroptosis (6) ────────────────
    # Activated/induced — not constitutively high in healthy hepatocytes.
    "TP53",      # p53 — stress sensor; elevated in MASH
    "CASP3",     # effector caspase — apoptosis executor; activated
    "RIPK3",     # necroptosis kinase; elevated in MASH
    "MLKL",      # necroptosis executor
    "GSDMD",     # gasdermin D — pyroptosis pore; specifically induced
    "BCL2L11",   # BIM — pro-apoptotic BH3-only; regulated

    # ── Bile acid / CYP enzymes (4) ───────────────────────────────────────────
    # Regulated by disease state — not constitutively maximal.
    "CYP7A1",    # rate-limiting BA synthesis; reduced in MASH
    "CYP2E1",    # metabolises FFAs → ROS; upregulated in MASH
    "ABCB11",    # BSEP — bile salt export; reduced in MASH
    "FGF19",     # ileal FXR hormone; steatosis modulator; regulated

    # ── ER stress / UPR (4) ──────────────────────────────────────────────────
    # Near-absent in healthy unstressed hepatocytes; induced by lipotoxicity.
    "DDIT3",     # CHOP — pro-apoptotic; strongly stress-induced in MASH
    "ATF3",      # hub gene MASH/ferroptosis networks (Lin 2025)
    "XBP1",      # IRE1α substrate; UPR TF; spliced under ER stress
    "HSPA5",     # GRP78/BiP — ER chaperone; upregulated in MASH

    # ── Autophagy / mitophagy (3) ─────────────────────────────────────────────
    "BECN1",     # beclin-1 — autophagy initiation; regulated
    "SQSTM1",    # p62 — aggregates in MASH hepatocytes; regulated
    "PINK1",     # mitophagy kinase; regulated by mitochondrial damage

    # ── NF-κB / JAK-STAT / MAPK inflammation hubs (4) ────────────────────────
    # Signalling molecules — activity/expression regulated by disease state.
    "NFKB1",     # NF-κB p50 — master inflammatory TF
    "STAT3",     # JAK-STAT; IL-6 effector; regulated
    "MAPK8",     # JNK1 — stress kinase; insulin resistance; MASH driver
    "JAK2",      # JAK-STAT kinase; regulated

    # ── Insulin resistance / glucose (2) ──────────────────────────────────────
    "INSR",      # insulin receptor — reduced signalling in MASH hepatocytes
    "IRS1",      # insulin receptor substrate 1; regulated

    # ── Novel scRNA-seq / spatial MASH drivers (4) ────────────────────────────
    # Identified from recent 2024-2025 MASH multi-omics studies.
    "EGR1",      # early growth response 1 — stress TF; MASH network hub
    "ZFP36",     # ZFP36/TTP — mRNA stability; anti-inflammatory; regulated
    "NAMPT",     # nicotinamide phosphoribosyltransferase; MASH-regulated
    "GADD45B",   # stress-inducible; MASH biomarker (nomogram study 2021)

]
_seen = set()
_deduped = []
for g in MASH_GENES:
    if g not in _seen:
        _deduped.append(g)
        _seen.add(g)
MASH_GENES = _deduped

print(f"✅ {len(MASH_GENES)} unique literature-curated MASH/liver candidate genes loaded.")
print(f"   (Single shared list — same genes tested across ALL cell types)")


In [ ]:
import os, pickle, numpy as np, pandas as pd
import datasets as hf_datasets

# ── Map MASH_GENES symbols → Ensembl IDs via the combined var table ──────────
all_var = pd.concat([a.var for a in adatas])
sym_to_ensembl = dict(zip(all_var.index, all_var["ensembl_id"]))

def get_geneformer_vocabulary():
    vocab_path = "/content/Geneformer/geneformer/gene_median_dictionary.pkl"
    if os.path.exists(vocab_path):
        with open(vocab_path, "rb") as f:
            return set(pickle.load(f).keys())
    print("   ⚠️  Gene median dictionary not found — skipping vocab filter.")
    return None

gf_vocab = get_geneformer_vocabulary()

# ── Build the shared candidate DataFrame ─────────────────────────────────────
# Maps each gene symbol to its Ensembl ID and Geneformer token ID.
# Genes not in the Geneformer vocabulary or not mappable are dropped.
rows = []
missing_ensembl, missing_token = [], []
for symbol in MASH_GENES:
    eid = sym_to_ensembl.get(symbol)
    if eid is None or (isinstance(eid, float) and np.isnan(eid)):
        missing_ensembl.append(symbol)
        continue
    if gf_vocab and eid not in gf_vocab:
        missing_token.append(symbol)
        continue
    token_id = tk.gene_token_dict.get(eid)
    if token_id is None:
        missing_token.append(symbol)
        continue
    rows.append({"gene_symbol": symbol, "ensembl_id": eid, "token_id": token_id,
                 "n_tokenized_cells": 0})

shared_candidate_df = pd.DataFrame(rows)
print(f"\n✅ Shared candidate gene mapping:")
print(f"   Input genes        : {len(MASH_GENES)}")
print(f"   Mapped to Ensembl  : {len(shared_candidate_df)}")
print(f"   No Ensembl ID      : {len(missing_ensembl)}  {missing_ensembl[:5]}")
print(f"   Not in GF vocab    : {len(missing_token)}   {missing_token[:5]}")

# ── Count token presence per cell type (for reporting only) ──────────────────
# This does NOT filter the candidate list — all mapped genes are tested.
DATASET_PATH = os.path.join(TOKENIZED_DATA_DIR, "geneformer_input.dataset")
tok_dataset = hf_datasets.load_from_disk(DATASET_PATH)

candidate_token_set = set(shared_candidate_df["token_id"].tolist())
ct_token_counts = {ct: {} for ct in passing_celltypes}

def count_tokens_batch(batch):
    for input_ids, ct in zip(batch["input_ids"], batch["cell_type"]):
        if ct not in ct_token_counts:
            return batch
        for tid in set(input_ids):
            ct_token_counts[ct][tid] = ct_token_counts[ct].get(tid, 0) + 1
    return batch

ctrl_dataset = tok_dataset.filter(
    lambda x: x["condition"] == CONTROL_LABEL and
              x["cell_type"] in set(passing_celltypes) and
              x["assay_type"] == "scRNA-seq",
    num_proc=4, batch_size=1000
)
ctrl_dataset.map(count_tokens_batch, batched=True, batch_size=100, num_proc=1)

# Report coverage per cell type — same candidate_df used for all
print("\nCandidate gene token presence across cell types:")
for ct in passing_celltypes:
    total_cells = (pd.Series(ctrl_dataset["cell_type"]) == ct).sum()
    present = sum(
        1 for row in shared_candidate_df.itertuples()
        if ct_token_counts.get(ct, {}).get(row.token_id, 0) >= 5
    )
    print(f"   {ct:45s}: {present}/{len(shared_candidate_df)} genes "
          f"in ≥5 cells  (total cells={total_cells})")

# ── Build candidate_dfs: same DataFrame for every passing cell type ───────────
candidate_dfs = {ct: shared_candidate_df.copy() for ct in passing_celltypes}

shared_candidate_df.to_csv(os.path.join(OUTPUT_DIR, "candidate_genes_shared.csv"), index=False)
print(f"\n✅ candidate_dfs built: {len(candidate_dfs)} cell types, "
      f"{len(shared_candidate_df)} shared candidate genes each.")
print("   Saved: candidate_genes_shared.csv")


## 9.In-Silico Perturbation with Geneformer

In [ ]:
import torch
import os

# Verify CUDA is available and set device explicitly
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device count: {torch.cuda.device_count()}')

# Force all torch operations to GPU
torch.cuda.set_device(0)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Verify
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Warm up the GPU — this forces CUDA initialisation before ISP
_ = torch.zeros(1).cuda()
print(f'GPU warmed up: {torch.cuda.get_device_name(0)}')
print(f'GPU memory after warmup: '
      f'{torch.cuda.memory_allocated()/1e9:.2f}GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f}GB reserved')

In [ ]:
# Run this once BEFORE the ISP cell
import inspect
import geneformer.in_silico_perturber as isp_module

src = inspect.getsource(isp_module)
device_lines = [
    (i, line.strip()) for i, line in enumerate(src.split('\n'))
    if 'device' in line.lower() and
    any(x in line.lower() for x in ['cuda', 'cpu', 'torch.device', '.to('])
]
for lineno, line in device_lines[:30]:
    print(f'  {lineno:4d}: {line}')

In [ ]:
import sys
if "/content/Geneformer" not in sys.path:
    sys.path.insert(0, "/content/Geneformer")
import os, hashlib, shutil, tempfile, subprocess, pandas as pd
import geneformer.perturber_utils as pu
import datasets as hf_datasets
from geneformer import InSilicoPerturber
import torch
import numpy as np


# ══════════════════════════════════════════════════════════════════════════════
# GPU SETUP
# ══════════════════════════════════════════════════════════════════════════════

def setup_gpu():
    if not torch.cuda.is_available():
        print("⚠️  CUDA not available — ISP will run on CPU (very slow).")
        return torch.device("cpu")

    device = torch.device("cuda:0")
    torch.cuda.set_device(0)

    # expandable_segments prevents memory fragmentation during forward passes.
    # Note: max_split_size_mb is intentionally omitted — it was only needed
    # alongside torch.compile which is disabled below.
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["TOKENIZERS_PARALLELISM"]   = "false"

    # Warm up CUDA context so the first cell type doesn't pay init overhead
    _ = torch.zeros(1, device=device)
    torch.cuda.synchronize()

    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}  ({vram_gb:.0f} GB VRAM)")

    try:
        util = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=utilization.gpu,memory.used,power.draw",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        print(f"   GPU status: {util}")
    except Exception:
        pass

    return device


DEVICE = setup_gpu()


# ══════════════════════════════════════════════════════════════════════════════
# PATCH pu.load_model — force GPU placement
#
# Guard prevents recursive patching if this cell is re-run.
#
# torch.compile is intentionally disabled:
#   - reduce-overhead mode uses CUDA Graphs which pre-allocate ~20 GB of
#     private memory pools, causing OOM with full-length sequences.
#   - default mode also triggers CUDA Graph state corruption after OOM,
#     producing AssertionError in cudagraph_trees.py on subsequent runs.
#   - inference_mode + CUDAPrefetchStream (below) provide sufficient speedup
#     without any CUDA Graph involvement.
# ══════════════════════════════════════════════════════════════════════════════

if not getattr(pu, "_load_model_gpu_patched", False):
    # Capture the real original function object directly — NOT via pu.load_model
    # attribute access. After pu.load_model is reassigned, _REAL_LOAD_MODEL
    # still points at the original, preventing infinite recursion.
    _REAL_LOAD_MODEL = pu.load_model

    def _gpu_load_model(model_type, num_classes, model_directory,
                         *args, **kwargs):
        """
        Wraps pu.load_model to guarantee GPU placement after loading.
        Calls _REAL_LOAD_MODEL directly — never pu.load_model — to prevent
        the recursion error that occurs when the patched function calls
        itself via the pu.load_model attribute.
        """
        model = _REAL_LOAD_MODEL(
            model_type, num_classes, model_directory, *args, **kwargs
        )
        if torch.cuda.is_available():
            model = model.to("cuda:0")
            model = model.eval()   # disables dropout; speeds up inference
            print(f"   Model device: {next(model.parameters()).device}")
            # torch.compile intentionally skipped — see note above
            alloc_gb  = torch.cuda.memory_allocated() / 1e9
            reserv_gb = torch.cuda.memory_reserved()  / 1e9
            print(f"   VRAM after model load: "
                  f"{alloc_gb:.2f} GB allocated, "
                  f"{reserv_gb:.2f} GB reserved")
        return model

    pu.load_model = _gpu_load_model
    pu._load_model_gpu_patched = True
    print("✅ pu.load_model patched for GPU (once per session).")
else:
    print("✅ pu.load_model already patched — skipping re-patch.")


# ══════════════════════════════════════════════════════════════════════════════
# CUDA PREFETCH STREAM — overlaps CPU data preparation with GPU inference
#
# While the GPU processes batch N, the CPU simultaneously prepares and
# transfers batch N+1 via a separate CUDA stream. This eliminates the
# CPU-GPU pipeline stall that causes low GPU utilisation at batch_size=64.
# ══════════════════════════════════════════════════════════════════════════════

class CUDAPrefetchStream:
    def __init__(self, iterable, device="cuda:0"):
        self.iterable = iterable
        self.device   = device
        self.stream   = torch.cuda.Stream(device=device)
        self._next    = None

    def _to_device(self, obj):
        if isinstance(obj, torch.Tensor):
            return obj.to(self.device, non_blocking=True)
        if isinstance(obj, dict):
            return {k: self._to_device(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return type(obj)(self._to_device(v) for v in obj)
        return obj

    def _preload(self):
        try:
            item = next(self._iter)
        except StopIteration:
            self._next = None
            return
        with torch.cuda.stream(self.stream):
            self._next = self._to_device(item)

    def __iter__(self):
        self._iter = iter(self.iterable)
        self._preload()
        return self

    def __next__(self):
        torch.cuda.current_stream().wait_stream(self.stream)
        item = self._next
        if item is None:
            raise StopIteration
        self._preload()
        return item

    def __len__(self):
        try:
            return len(self.iterable)
        except TypeError:
            return 0


_OriginalDataLoader = torch.utils.data.DataLoader

if torch.cuda.is_available() and not getattr(
        torch.utils.data.DataLoader, "_prefetch_patched", False):

    class _PrefetchingDataLoader(_OriginalDataLoader):
        _prefetch_patched = True

        def __iter__(self):
            return iter(
                CUDAPrefetchStream(super().__iter__(), device="cuda:0")
            )

    torch.utils.data.DataLoader = _PrefetchingDataLoader
    print("✅ DataLoader patched with CUDA prefetch stream.")
else:
    if not torch.cuda.is_available():
        print("   DataLoader prefetch: skipped (no CUDA).")
    else:
        print("   DataLoader prefetch: already patched.")


# ══════════════════════════════════════════════════════════════════════════════
# PATIENT-AWARE CELL SAMPLING (unchanged — working correctly)
# ══════════════════════════════════════════════════════════════════════════════

def patient_aware_sample_for_isp(
        dataset, donor_key="donor",
        max_cells_total=500, min_cells_per_donor=5,
        random_state=42):
    """
    Donor-stratified sampling of cell indices from a pre-filtered HuggingFace
    Dataset. Returns (sampled_indices, donor_report).
    """
    rng = np.random.default_rng(random_state)

    if donor_key not in dataset.column_names:
        print(f"   ⚠️  donor_key '{donor_key}' not found — random sampling.")
        n = min(max_cells_total, len(dataset))
        return rng.choice(len(dataset), size=n, replace=False).tolist(), \
               {"all_cells": n}

    donors = dataset[donor_key]
    donor_to_indices = {}
    for i, d in enumerate(donors):
        donor_to_indices.setdefault(d, []).append(i)

    eligible = {d: idxs for d, idxs in donor_to_indices.items()
                if len(idxs) >= min_cells_per_donor}

    if not eligible:
        print(f"   ⚠️  No donors with ≥{min_cells_per_donor} cells "
              f"— random sampling.")
        n = min(max_cells_total, len(dataset))
        return rng.choice(len(dataset), size=n, replace=False).tolist(), \
               {"all_cells": n}

    n_donors = len(eligible)
    cells_per_donor = max(min_cells_per_donor,
                          int(np.floor(max_cells_total / n_donors)))
    sampled, report = [], {}
    for donor, idxs in eligible.items():
        n_take = min(cells_per_donor, len(idxs))
        chosen = rng.choice(idxs, size=n_take, replace=False).tolist()
        sampled.extend(chosen)
        report[donor] = n_take

    if len(sampled) > max_cells_total:
        sampled = rng.choice(sampled, size=max_cells_total,
                             replace=False).tolist()

    print(f"   Patient-aware sampling: {len(sampled)} cells from "
          f"{n_donors} donors (~{cells_per_donor} cells/donor)")
    return sampled, report



def trim_sequences_safe(dataset, max_len=1024, protected_token_ids=None):
    """
    Trim token sequences to max_len while guaranteeing that all candidate
    gene tokens AND Geneformer special tokens are never removed.
    """
    if protected_token_ids is None:
        protected_token_ids = set()

    lengths   = dataset["length"]
    max_orig  = int(np.max(lengths))
    mean_orig = float(np.mean(lengths))

    if max_len >= max_orig:
        print(f"   max_len={max_len} >= max sequence length {max_orig} "
              f"— no trimming needed.")
        return dataset

    # ── Detect Geneformer special tokens from the dataset ────────────────
    # CLS token is always at position 0, EOS token is always at the end.
    # We must never remove these or Geneformer's internal loop will crash.
    sample_ids  = dataset["input_ids"][0]
    cls_token   = int(sample_ids[0])   # always first
    eos_token   = int(sample_ids[-1])  # always last

    # Add special tokens to the protected set so the swap logic
    # never overwrites them and the trim never removes them
    fully_protected = protected_token_ids | {cls_token, eos_token}

    print(f"   Special tokens detected — CLS: {cls_token}, EOS: {eos_token}")

    n_trimmed = sum(1 for l in lengths if l > max_len)
    pct       = (1 - max_len / max_orig) * 100
    print(f"   Trimming {n_trimmed}/{len(dataset)} sequences to {max_len} "
          f"tokens ({pct:.0f}% reduction from max {max_orig}). "
          f"Mean: {mean_orig:.0f} → ≤{max_len}.")

    def trim_example(example):
        ids = list(example["input_ids"])
        if len(ids) <= max_len:
            return example

        # Always keep CLS at position 0 and EOS at the last position —
        # extract them, trim the middle, then reattach
        cls = ids[0]
        eos = ids[-1]
        middle = ids[1:-1]   # gene tokens only, no special tokens

        # Target middle length: max_len - 2 (for CLS + EOS)
        target_middle = max_len - 2

        if len(middle) <= target_middle:
            return example

        keep_middle = middle[:target_middle]
        tail_middle = middle[target_middle:]

        # Find candidate tokens that ended up in the tail
        protected_in_tail = [
            tid for tid in tail_middle
            if int(tid) in protected_token_ids   # candidate genes only
        ]

        if protected_in_tail:
            # Swap protected tail tokens into the keep window,
            # replacing the lowest-ranked non-protected positions
            # (working from the end of keep_middle backward)
            swap_positions = [
                i for i in range(len(keep_middle) - 1, -1, -1)
                if int(keep_middle[i]) not in fully_protected
            ]
            for swap_pos, prot_tid in zip(swap_positions, protected_in_tail):
                keep_middle[swap_pos] = prot_tid

        # Reconstruct: CLS + trimmed gene tokens + EOS
        example["input_ids"] = [cls] + keep_middle + [eos]
        example["length"]    = len(example["input_ids"])
        return example

    trimmed = dataset.map(trim_example, num_proc=1)

    # Verify special tokens are intact in a sample
    sample_trimmed = trimmed["input_ids"][0]
    assert int(sample_trimmed[0])  == cls_token, "CLS token missing after trim"
    assert int(sample_trimmed[-1]) == eos_token, "EOS token missing after trim"
    print(f"   ✅ Special tokens intact. "
          f"Sample length after trim: {len(sample_trimmed)}")

    return trimmed
# ══════════════════════════════════════════════════════════════════════════════
# PatchedISP — candidate gene filtering, full sequences, GPU-optimised
# ══════════════════════════════════════════════════════════════════════════════

class PatchedISP(InSilicoPerturber):
    def __init__(self, candidate_token_ids=None, **kwargs):
        kwargs["genes_to_perturb"] = "all"
        self.candidate_token_ids = (
            set(int(t) for t in candidate_token_ids)
            if candidate_token_ids else None
        )
        super().__init__(**kwargs)

    def apply_additional_filters(self, filtered_input_data):
        if self.cell_states_to_model is not None:
            filtered_input_data = pu.filter_data_by_start_state(
                filtered_input_data, self.cell_states_to_model, self.nproc)
        if self.anchor_token is not None:
            filtered_input_data = pu.filter_data_by_tokens_and_log(
                filtered_input_data, self.anchor_token, self.nproc,
                "anchor_gene")
        filtered_input_data = pu.downsample_and_sort(
            filtered_input_data, self.max_ncells)
        if self.cell_inds_to_perturb != "all":
            filtered_input_data = pu.slice_by_inds_to_perturb(
                filtered_input_data, self.cell_inds_to_perturb)
        return filtered_input_data

    def perturb_data(self, model_directory, input_data_file,
                     output_directory, output_prefix):
        # torch.inference_mode disables autograd tracking for the entire run —
        # reduces memory overhead and speeds up each forward pass.
        # No CUDA Graphs are involved so there is no OOM risk.
        with torch.inference_mode():
            if self.candidate_token_ids:
                print("   Filtering to cells containing ≥1 candidate gene "
                      "token (full sequences preserved)...")
                full_dataset  = hf_datasets.load_from_disk(
                    input_data_file, keep_in_memory=True
                )
                candidate_set = self.candidate_token_ids

                def has_candidate(example):
                    return any(int(t) in candidate_set
                               for t in example["input_ids"])

                filtered_dataset = full_dataset.filter(
                    has_candidate, num_proc=1
                )
                n_before = len(full_dataset)
                n_after  = len(filtered_dataset)
                print(f"   {n_after}/{n_before} cells contain "
                      f"≥1 candidate gene token.")

                if n_after == 0:
                    print("   ⚠️  No cells remain. Skipping.")
                    return

                # Compact into a single contiguous Arrow file —
                # faster sequential reads during the perturbation loop
                filtered_dataset = filtered_dataset.flatten_indices()

                tmp_dir  = tempfile.mkdtemp()
                tmp_path = os.path.join(tmp_dir, "candidate_cells.dataset")
                filtered_dataset.save_to_disk(tmp_path)

                try:
                    util = subprocess.run(
                        ["nvidia-smi",
                         "--query-gpu=utilization.gpu,memory.used,power.draw",
                         "--format=csv,noheader"],
                        capture_output=True, text=True, timeout=5
                    ).stdout.strip()
                    # print(f"   GPU before forward passes: {util}")
                    # # Replace this single line:
                    # print(f"   GPU before forward passes: {util}")

                    # With this background monitor that logs every 30s during ISP:
                    # import threading

                    # def _gpu_monitor(stop_event, interval=30):
                    #     while not stop_event.is_set():
                    #         try:
                    #             out = subprocess.run(
                    #                 ["nvidia-smi",
                    #                 "--query-gpu=utilization.gpu,memory.used,power.draw",
                    #                 "--format=csv,noheader"],
                    #                 capture_output=True, text=True, timeout=5
                    #             ).stdout.strip()
                    #             print(f"   [GPU monitor] {out}", flush=True)
                    #         except Exception:
                    #             pass
                    #         stop_event.wait(interval)

                    # _stop = threading.Event()
                    # _monitor_thread = threading.Thread(
                    #     target=_gpu_monitor, args=(_stop,), daemon=True
                    # )
                    # _monitor_thread.start()
                    # print(f"   GPU before forward passes: {util}")

                    # try:
                    #     super().perturb_data(
                    #         model_directory, tmp_path,
                    #         output_directory, output_prefix
                    #     )
                    # finally:
                    #     _stop.set()
                    #     _monitor_thread.join(timeout=5)
                    #     shutil.rmtree(tmp_dir, ignore_errors=True)
                    #     torch.cuda.empty_cache()
                except Exception:
                    pass

                try:
                    super().perturb_data(
                        model_directory, tmp_path,
                        output_directory, output_prefix
                    )
                finally:
                    shutil.rmtree(tmp_dir, ignore_errors=True)
                    torch.cuda.empty_cache()
            else:
                super().perturb_data(
                    model_directory, input_data_file,
                    output_directory, output_prefix
                )
            torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════════════════
# MAIN ISP LOOP
# ══════════════════════════════════════════════════════════════════════════════

ISP_OUTPUT_DIR  = os.path.join(OUTPUT_DIR, "isp_output")
os.makedirs(ISP_OUTPUT_DIR, exist_ok=True)
MODE_MAP        = {"down": "delete", "delete": "delete", "up": "overexpress"}
gf_perturb_type = MODE_MAP.get(PERTURB_MODE, "delete")
DATASET_PATH    = os.path.join(TOKENIZED_DATA_DIR, "geneformer_input.dataset")

print(f"\n⚡ Running Geneformer ISP (GPU-optimised, full token sequences)...")
print(f"   Device            : {DEVICE}")
print(f"   Perturbation mode : {gf_perturb_type}")
print(f"   Shared candidates : {len(shared_candidate_df)} genes")
print(f"   Cell types        : {list(candidate_dfs.keys())}")
print(f"   Control → Target  : {CONTROL_LABEL} → {TARGET_LABEL}")

original_write = pu.write_perturbation_dictionary

def safe_write(data, path):
    basename = os.path.basename(path)
    if len(basename) > 200:
        dir_part    = os.path.dirname(path)
        safe_prefix = basename[:60]
        token_hash  = hashlib.md5(basename.encode()).hexdigest()[:16]
        path        = os.path.join(dir_part, f"{safe_prefix}_{token_hash}")
    original_write(data, path)

pu.write_perturbation_dictionary = safe_write

import time

try:
    for cell_type, candidate_df in candidate_dfs.items():
        t_cell_start = time.time()
        print(f"\n{'─'*60}")
        print(f"Cell type: {cell_type}")

        # ── State embeddings ──────────────────────────────────────────────
        ctrl_mask = ((embeddings["cell_type"] == cell_type) &
                     (embeddings["condition"] == CONTROL_LABEL))
        tgt_mask  = ((embeddings["cell_type"] == cell_type) &
                     (embeddings["condition"] == TARGET_LABEL))

        if ctrl_mask.sum() == 0 or tgt_mask.sum() == 0:
            print(f"   ⚠️  Skipping: insufficient cells for state embeddings.")
            continue

        control_embs = embeddings.loc[ctrl_mask, emb_feature_cols].values
        target_embs  = embeddings.loc[tgt_mask,  emb_feature_cols].values

        state_embs_dict_for_isp = {
            CONTROL_LABEL: torch.tensor(
                control_embs.mean(axis=0), dtype=torch.float32
            ).to(DEVICE),
            TARGET_LABEL: torch.tensor(
                target_embs.mean(axis=0), dtype=torch.float32
            ).to(DEVICE),
        }

        # ── Candidate token IDs ───────────────────────────────────────────
        candidate_token_ids = [
            int(row.token_id) for row in candidate_df.itertuples()
            if hasattr(row, "token_id") and row.token_id is not None
        ]
        if not candidate_token_ids:
            candidate_token_ids = [
                int(tk.gene_token_dict[eid])
                for eid in candidate_df["ensembl_id"]
                if eid in tk.gene_token_dict
            ]
        print(f"   {len(candidate_token_ids)} candidate token IDs.")

        ct_isp_dir = os.path.join(ISP_OUTPUT_DIR, cell_type.replace(" ", "_"))
        os.makedirs(ct_isp_dir, exist_ok=True)

        # ── Step 1: Load and filter to this cell type / condition ─────────
        full_dataset    = hf_datasets.load_from_disk(DATASET_PATH)
        ct_cond_dataset = full_dataset.filter(
            lambda x: (x["cell_type"] == cell_type and
                       x["condition"] == CONTROL_LABEL and
                       x["assay_type"] == "scRNA-seq"),
            num_proc=1
        )
        print(f"   {len(ct_cond_dataset)} control scRNA-seq cells available.")

        # ── Step 2: Detect donor column ───────────────────────────────────
        _donor_key_isp = None
        for _dk in ["donor", "Donor", "donor_id", "patient_id", "sample_id"]:
            if _dk in ct_cond_dataset.column_names:
                _donor_key_isp = _dk
                print(f"   Donor column: \"{_donor_key_isp}\"")
                break

        # ── Step 3: Patient-aware sampling (unchanged) ────────────────────
        if _donor_key_isp and len(ct_cond_dataset) > 0:
            sampled_idx, donor_report = patient_aware_sample_for_isp(
                ct_cond_dataset,
                donor_key=_donor_key_isp,
                max_cells_total=MAX_ISP_CELLS_TOTAL,
                min_cells_per_donor=MIN_CELLS_PER_DONOR_ISP,
            )
            print(f"   Donor report: {donor_report}")
            pd.DataFrame(
                list(donor_report.items()), columns=["donor", "n_cells"]
            ).to_csv(
                os.path.join(
                    ct_isp_dir,
                    f"patient_sampling_{cell_type.replace(' ', '_')}.csv"
                ),
                index=False
            )
        else:
            print(f"   ⚠️  No donor column — random sampling "
                  f"(max {MAX_ISP_CELLS_TOTAL}).")
            rng         = np.random.default_rng(42)
            n_take      = min(MAX_ISP_CELLS_TOTAL, len(ct_cond_dataset))
            sampled_idx = rng.choice(
                len(ct_cond_dataset), size=n_take, replace=False
            ).tolist()
            donor_report = {"random_fallback": n_take}

        isp_input_dataset = ct_cond_dataset.select(sampled_idx)
        print(f"   ISP input: {len(isp_input_dataset)} patient-balanced cells.")

        # ── Step 4: Pre-filter to cells containing candidate genes ────────
        candidate_set_int = set(int(t) for t in candidate_token_ids)

        def _has_candidate(example):
            return any(int(t) in candidate_set_int
                       for t in example["input_ids"])

        isp_input_dataset = isp_input_dataset.filter(
            _has_candidate, num_proc=1
        )
        print(f"   After candidate filter: {len(isp_input_dataset)} cells.")

        if len(isp_input_dataset) == 0:
            print(f"   ⚠️  No cells with candidate genes — "
                  f"skipping {cell_type}.")
            continue

        isp_input_dataset = trim_sequences_safe(
            isp_input_dataset,
            max_len=1024,
            protected_token_ids=set(int(t) for t in candidate_token_ids),
        )

        # ── Step 5: Compact and save to tmp dir ───────────────────────────
        isp_input_dataset = isp_input_dataset.flatten_indices()

        isp_tmp_dir    = tempfile.mkdtemp(prefix="isp_gpu_")
        isp_input_path = os.path.join(isp_tmp_dir, "patient_balanced.dataset")
        isp_input_dataset.save_to_disk(isp_input_path)

        try:
            # ── Step 6: Run ISP ───────────────────────────────────────────
            cell_states_to_model_for_isp = {
                "state_key":   "condition",
                "start_state": CONTROL_LABEL,
                "goal_state":  TARGET_LABEL,
                "alt_states":  []
            }

            isp = PatchedISP(
                candidate_token_ids=candidate_token_ids,
                perturb_type=gf_perturb_type, perturb_rank_shift=None,
                combos=0, anchor_gene=None,
                model_type="Pretrained", num_classes=0,
                emb_mode="cls", cell_emb_style="mean_pool",
                filter_data={"cell_type": [cell_type],
                             "condition": [CONTROL_LABEL],
                             "assay_type": ["scRNA-seq"]},
                cell_states_to_model=cell_states_to_model_for_isp,
                state_embs_dict=state_embs_dict_for_isp,
                max_ncells=None,
                emb_layer=-1,
                forward_batch_size=64,
                nproc=1,
            )

            isp.perturb_data(
                model_directory="ctheodoris/Geneformer",
                input_data_file=isp_input_path,
                output_directory=ct_isp_dir,
                output_prefix=f"isp_{cell_type.replace(' ', '_')}"
            )

            elapsed = (time.time() - t_cell_start) / 60
            print(f"   ✅ ISP complete for {cell_type} ({elapsed:.1f} min).")

            try:
                util = subprocess.run(
                    ["nvidia-smi",
                     "--query-gpu=utilization.gpu,memory.used,power.draw",
                     "--format=csv,noheader"],
                    capture_output=True, text=True, timeout=5
                ).stdout.strip()
                print(f"   GPU after {cell_type}: {util}")
            except Exception:
                pass

        finally:
            shutil.rmtree(isp_tmp_dir, ignore_errors=True)
            torch.cuda.empty_cache()

finally:
    pu.write_perturbation_dictionary = original_write
    if torch.cuda.is_available():
        torch.utils.data.DataLoader = _OriginalDataLoader
        print("   DataLoader restored.")

print("\n✅ All ISP runs complete.")

## 10.Compute Cosine Shift & Statistical Testing (following Liu et al.)

In [ ]:
import sys
if "/content/Geneformer" not in sys.path:
    sys.path.insert(0, "/content/Geneformer")
import os, pickle, pandas as pd, numpy as np, torch
from collections import defaultdict
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ── Liu et al. ISP statistical method ────────────────────────────────────────
# Pickle files store raw cosine similarities:
#   cos_sim(perturbed_cell_emb, target_centroid)
#
# Cosine shift for gene g in cell i:
#   shift(g,i) = cos_sim(perturbed_i, target_centroid) - baseline_cos_sim
#
# where baseline_cos_sim = cos_sim(control_centroid, target_centroid).
#
# Statistical test: Wilcoxon rank-sum, Sample A (gene shifts) vs Sample B
# (cross-gene random baseline, floored at 0 if median < 0). BH correction.
# ─────────────────────────────────────────────────────────────────────────────

STATS_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "isp_stats")
os.makedirs(STATS_OUTPUT_DIR, exist_ok=True)
all_results = []

token_to_ensembl = {v: k for k, v in tk.gene_token_dict.items()}

# Lower MIN_CELLS_PER_GENE to 5 — sequence trimming reduces the number of
# cells some genes appear in, and 10 is too strict for trimmed sequences.
# Genes with <5 cells still get a row (with pval_adj=1, significant=False)
# so they appear in non-significant plots rather than being silently absent.
MIN_CELLS_PER_GENE    = 20
EFFECT_SIZE_THRESHOLD = 2.5e-4

rng = np.random.default_rng(42)

for cell_type in candidate_dfs.keys():
    print(f"\n{'='*60}")
    print(f"Stats for: {cell_type}")

    candidate_df = candidate_dfs[cell_type]
    candidate_ensembl_set = set(candidate_df["ensembl_id"].dropna().tolist())
    candidate_token_ids_set = {
        tk.gene_token_dict[eid]
        for eid in candidate_ensembl_set
        if eid in tk.gene_token_dict
    }

    ct_isp_dir = os.path.join(ISP_OUTPUT_DIR, cell_type.replace(" ", "_"))
    pickle_files = [
        f for f in os.listdir(ct_isp_dir)
        if f.endswith(".pickle") and "dict_cell_embs_" in f
    ]
    print(f"   {len(pickle_files)} pickle files found.")
    if not pickle_files:
        print("   ⚠️  No pickle files — skipping.")
        continue

    # ── Load and merge all pickle files ──────────────────────────────────
    merged_by_state = defaultdict(lambda: defaultdict(list))
    n_loaded = n_failed = 0
    for fname in pickle_files:
        fpath = os.path.join(ct_isp_dir, fname)
        try:
            with open(fpath, "rb") as f:
                batch_dict = pickle.load(f)
            if isinstance(batch_dict, list):
                batch_dict = batch_dict[0] if batch_dict else {}
            for state_label, inner_dict in batch_dict.items():
                for key, val in inner_dict.items():
                    if isinstance(val, list):
                        merged_by_state[state_label][key].extend(val)
                    else:
                        merged_by_state[state_label][key].append(val)
            n_loaded += 1
        except Exception as e:
            print(f"   ⚠️  Failed to load {fname}: {e}")
            n_failed += 1

    print(f"   Loaded {n_loaded} files ({n_failed} failed).")
    print(f"   States in pickles: {list(merged_by_state.keys())}")

    if TARGET_LABEL not in merged_by_state:
        print(f"   ⚠️  Target state '{TARGET_LABEL}' not found. "
              f"Available: {list(merged_by_state.keys())}")
        continue

    target_dict = merged_by_state[TARGET_LABEL]

    # ── Compute baseline cosine similarity ───────────────────────────────
    # baseline_cos_sim = cos_sim(control_centroid, target_centroid)
    # This is subtracted from every raw pickle value to convert
    # raw cosine similarities (~0.9) into shifts (~±0.001–0.05).
    ctrl_mask = ((embeddings["cell_type"] == cell_type) &
                 (embeddings["condition"] == CONTROL_LABEL))
    tgt_mask  = ((embeddings["cell_type"] == cell_type) &
                 (embeddings["condition"] == TARGET_LABEL))

    ctrl_mean = embeddings.loc[ctrl_mask, emb_feature_cols].values.mean(axis=0)
    tgt_mean  = embeddings.loc[tgt_mask,  emb_feature_cols].values.mean(axis=0)
    ctrl_norm = ctrl_mean / (np.linalg.norm(ctrl_mean) + 1e-12)
    tgt_norm  = tgt_mean  / (np.linalg.norm(tgt_mean)  + 1e-12)
    baseline_cos_sim = float(np.dot(ctrl_norm, tgt_norm))
    print(f"   Baseline cos sim (ctrl → tgt centroid): {baseline_cos_sim:.4f}")

    # ── Diagnose raw pickle values BEFORE subtraction ─────────────────────
    # This confirms whether the pickles store raw cos sims (near 0.9)
    # or pre-computed shifts (near 0.0). Subtraction is only correct
    # when pickles store raw cos sims.
    _sample_key = next(
        ((tid, ek) for (tid, ek) in target_dict.keys() if ek == "cell_emb"),
        None
    )
    if _sample_key:
        _raw_sample = np.array(
            target_dict[_sample_key][:10], dtype=np.float64
        )
        _raw_sample = _raw_sample[np.isfinite(_raw_sample)]
        _raw_median = float(np.median(_raw_sample)) if len(_raw_sample) else 0.0
        print(f"   Raw pickle sample (first gene, up to 10 cells): "
              f"{np.round(_raw_sample, 4).tolist()}")
        print(f"   Raw median: {_raw_median:.4f}  |  "
              f"baseline: {baseline_cos_sim:.4f}  |  "
              f"shift after subtraction: {_raw_median - baseline_cos_sim:.5f}")

        if abs(_raw_median) < 0.1:
            # Values near 0 → already shifts, do NOT subtract baseline
            print("   ⚠️  Raw values near 0 — pickles appear to store "
                  "PRE-COMPUTED shifts. Setting baseline_cos_sim=0 "
                  "to avoid double-subtraction.")
            baseline_cos_sim = 0.0
        elif abs(_raw_median - baseline_cos_sim) > 0.5:
            print("   ⚠️  After subtraction shifts exceed ±0.5 — "
                  "possible baseline mismatch. Check embeddings are "
                  "from the same model run as ISP pickles.")
        else:
            print(f"   ✅ Raw values look like cos sims. "
                  f"Shifts after subtraction: ~"
                  f"{_raw_median - baseline_cos_sim:.5f}")

    # ── Extract per-gene cosine shifts ────────────────────────────────────
    all_gene_shifts   = {}   # token_id → np.array of shifts (all genes)
    all_gene_n_cells  = {}   # token_id → n cells with valid data
    skipped_too_few   = 0

    for (token_id, emb_key), cos_sims in target_dict.items():
        if emb_key != "cell_emb":
            continue
        arr = np.array(cos_sims, dtype=np.float64)
        arr = arr[np.isfinite(arr)]
        all_gene_n_cells[token_id] = len(arr)

        if len(arr) < MIN_CELLS_PER_GENE:
            skipped_too_few += 1
            continue

        shifts = arr - baseline_cos_sim
        all_gene_shifts[token_id] = shifts

    if skipped_too_few:
        print(f"   {skipped_too_few} genes skipped (< {MIN_CELLS_PER_GENE} "
              f"valid cells).")

    candidate_gene_shifts = {
        tid: s for tid, s in all_gene_shifts.items()
        if tid in candidate_token_ids_set
    }
    other_gene_shifts = {
        tid: s for tid, s in all_gene_shifts.items()
        if tid not in candidate_token_ids_set
    }

    print(f"   Candidate genes with shift data : {len(candidate_gene_shifts)}"
          f" / {len(candidate_token_ids_set)}")
    print(f"   Other genes (random baseline)   : {len(other_gene_shifts)}")

    # ── Check shift range ─────────────────────────────────────────────────
    if candidate_gene_shifts:
        _all_s = np.concatenate(list(candidate_gene_shifts.values()))
        _all_s = _all_s[np.isfinite(_all_s)]
        print(f"   Candidate shift range: "
              f"min={_all_s.min():.5f}, "
              f"max={_all_s.max():.5f}, "
              f"median={np.median(_all_s):.5f}")
        if _all_s.max() > 0.5 or _all_s.min() < -0.5:
            print("   ⚠️  Shifts outside ±0.5 — baseline subtraction "
                  "may be incorrect.")

    # ── Build complete candidate gene list including low-cell genes ───────
    # Genes with < MIN_CELLS_PER_GENE cells still need a row so they appear
    # in the non-significant distribution in downstream plots rather than
    # being silently absent.
    all_candidate_token_ids = candidate_token_ids_set

    if not candidate_gene_shifts and not all_candidate_token_ids:
        print("   ⚠️  No candidate gene data — skipping.")
        continue

    # ── Build cross-gene random baseline pool ────────────────────────────
    if other_gene_shifts:
        other_pool_all = np.concatenate(list(other_gene_shifts.values()))
        other_pool_all = other_pool_all[np.isfinite(other_pool_all)]
    else:
        print("   ⚠️  No other genes for baseline — using candidate pool.")
        other_pool_all = (
            np.concatenate(list(candidate_gene_shifts.values()))
            if candidate_gene_shifts else np.zeros(10)
        )
        other_pool_all = other_pool_all[np.isfinite(other_pool_all)]

    print(f"   Random baseline pool: n={len(other_pool_all)}, "
          f"median={np.median(other_pool_all):.5f}, "
          f"std={np.std(other_pool_all):.5f}")

    # ── Score each candidate gene ─────────────────────────────────────────
    rows = []

    for token_id in all_candidate_token_ids:
        ensembl_id = token_to_ensembl.get(token_id, str(token_id))
        match      = candidate_df.loc[
            candidate_df["ensembl_id"] == ensembl_id, "gene_symbol"
        ]
        gene_symbol = match.values[0] if len(match) > 0 else ensembl_id
        n_cells     = all_gene_n_cells.get(token_id, 0)

        if token_id in candidate_gene_shifts:
            # Gene has enough cells — compute full stats
            shifts_a = candidate_gene_shifts[token_id]
            n_a      = len(shifts_a)

            # Sample B: other-gene shifts excluding this gene
            other_for_this = np.concatenate([
                v for tid, v in other_gene_shifts.items()
                if tid != token_id
            ]) if other_gene_shifts else other_pool_all
            other_for_this = other_for_this[np.isfinite(other_for_this)]

            if len(other_for_this) >= n_a:
                sample_b = rng.choice(other_for_this, size=n_a, replace=False)
            elif len(other_for_this) > 0:
                sample_b = rng.choice(other_for_this, size=n_a, replace=True)
            else:
                sample_b = np.zeros(n_a)

            # Liu et al. floor: if random baseline median < 0, floor at 0
            baseline_was_floored = False
            if np.median(sample_b) < 0:
                sample_b = np.maximum(sample_b, 0.0)
                baseline_was_floored = True

            try:
                _, pval = stats.ranksums(shifts_a, sample_b)
            except Exception:
                pval = 1.0
            if not np.isfinite(pval):
                pval = 1.0

            median_shift = float(np.nanmedian(shifts_a))
            mean_shift   = float(np.nanmean(shifts_a))
            std_shift    = float(np.nanstd(shifts_a))

        else:
            # Gene has too few cells — write a row with neutral values
            # so it appears in non-significant plots rather than being absent
            pval                 = 1.0
            baseline_was_floored = False
            median_shift         = 0.0
            mean_shift           = 0.0
            std_shift            = 0.0

        rows.append({
            "gene_symbol":          gene_symbol,
            "ensembl_id":           ensembl_id,
            "median_cosine_shift":  median_shift,
            "mean_cosine_shift":    mean_shift,
            "std_cosine_shift":     std_shift,
            "median_cos_sim":       float(median_shift + baseline_cos_sim),
            "mean_cos_sim":         float(mean_shift   + baseline_cos_sim),
            "n_cells":              n_cells,
            "baseline_was_floored": baseline_was_floored,
            "pval_raw":             float(pval),
            "cell_type":            cell_type,
            "control_state":        CONTROL_LABEL,
            "target_state":         TARGET_LABEL,
            "perturb_mode":         PERTURB_MODE,
            "baseline_cos_sim":     baseline_cos_sim,
        })

    if not rows:
        print("   ⚠️  No rows to report.")
        continue

    results_df_ct = pd.DataFrame(rows)

    # ── NaN safety ────────────────────────────────────────────────────────
    # Fill any remaining NaN shifts with 0.0 so every gene has a row in
    # the non-significant distribution — prevents exclusion from plots
    results_df_ct["median_cosine_shift"] = (
        results_df_ct["median_cosine_shift"].fillna(0.0)
    )
    results_df_ct["mean_cosine_shift"] = (
        results_df_ct["mean_cosine_shift"].fillna(0.0)
    )
    results_df_ct["pval_raw"] = results_df_ct["pval_raw"].fillna(1.0)

    # ── BH correction ─────────────────────────────────────────────────────
    _, pval_adj, _, _ = multipletests(
        results_df_ct["pval_raw"], alpha=0.05, method="fdr_bh"
    )
    results_df_ct["pval_adj"] = pval_adj

    # ── Significance call ─────────────────────────────────────────────────
    # delete mode: negative shift = deletion moves cells away from target
    #              → this gene drives the disease state
    if gf_perturb_type == "delete":
        results_df_ct["significant"] = (
            (results_df_ct["pval_adj"] < 0.05) &
            (results_df_ct["median_cosine_shift"].abs() > EFFECT_SIZE_THRESHOLD) &
            (results_df_ct["median_cosine_shift"] < 0) &
            (results_df_ct["n_cells"] >= MIN_CELLS_PER_GENE)
        )
    else:
        results_df_ct["significant"] = (
            (results_df_ct["pval_adj"] < 0.05) &
            (results_df_ct["median_cosine_shift"].abs() > EFFECT_SIZE_THRESHOLD) &
            (results_df_ct["median_cosine_shift"] > 0) &
            (results_df_ct["n_cells"] >= MIN_CELLS_PER_GENE)
        )

    ascending = (gf_perturb_type == "delete")
    results_df_ct = results_df_ct.sort_values(
        "median_cosine_shift", ascending=ascending
    ).reset_index(drop=True)

    ct_stats_dir = os.path.join(STATS_OUTPUT_DIR, cell_type.replace(" ", "_"))
    os.makedirs(ct_stats_dir, exist_ok=True)
    out_csv = os.path.join(
        ct_stats_dir, f"stats_{cell_type.replace(' ', '_')}.csv"
    )
    results_df_ct.to_csv(out_csv, index=False)
    all_results.append(results_df_ct)

    n_sig   = results_df_ct["significant"].sum()
    n_total = len(results_df_ct)
    print(f"   ✅ {n_total} genes scored, {n_sig} significant "
          f"({n_sig/n_total*100:.1f}%).")
    if n_sig > 0:
        top_cols = ["gene_symbol", "median_cosine_shift",
                    "mean_cosine_shift", "pval_adj", "n_cells"]
        print(results_df_ct[results_df_ct["significant"]][top_cols]
              .head(15).to_string())

if all_results:
    final_df  = pd.concat(all_results, ignore_index=True)
    final_path = os.path.join(STATS_OUTPUT_DIR, "all_isp_results.csv")
    final_df.to_csv(final_path, index=False)
    print(f"\n✅ All results saved: {final_path}  ({len(final_df)} rows)")
else:
    print("\n⚠️  No results to save.")

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from collections import defaultdict
import pickle, os

# ── Bootstrap stability check ─────────────────────────────────────────────────
# Splits each cell type's ISP data into two independent random halves
# and checks whether the top hits are consistent across both halves.
# No ISP rerun required — uses existing pickle files directly.
# ─────────────────────────────────────────────────────────────────────────────

N_BOOTSTRAP_SPLITS = 10   # number of random splits to average over
SPLIT_FRACTION     = 0.5  # each half uses this fraction of cells
MIN_CELLS_HALF     = 20    # minimum cells per gene in each half

stability_results = {}
rng_boot = np.random.default_rng(123)

def score_gene_shifts(gene_shifts, other_pool, rng_obj,
                      effect_threshold=1e-4, perturb_type='delete'):
    """
    Run Liu et al. stats on a dict of gene_shifts.
    Returns a DataFrame with median_shift and significant columns.
    """
    rows = []
    for token_id, shifts_a in gene_shifts.items():
        if len(shifts_a) < MIN_CELLS_HALF:
            continue

        other_for_this = np.concatenate([
            v for tid, v in other_pool.items() if tid != token_id
        ]) if other_pool else np.zeros(len(shifts_a))
        other_for_this = other_for_this[np.isfinite(other_for_this)]

        if len(other_for_this) >= len(shifts_a):
            sample_b = rng_obj.choice(
                other_for_this, size=len(shifts_a), replace=False
            )
        elif len(other_for_this) > 0:
            sample_b = rng_obj.choice(
                other_for_this, size=len(shifts_a), replace=True
            )
        else:
            sample_b = np.zeros(len(shifts_a))

        if np.median(sample_b) < 0:
            sample_b = np.maximum(sample_b, 0.0)

        try:
            _, pval = stats.ranksums(shifts_a, sample_b)
        except Exception:
            pval = 1.0

        rows.append({
            'token_id':           token_id,
            'median_shift':       float(np.nanmedian(shifts_a)),
            'n_cells':            len(shifts_a),
            'pval_raw':           float(pval) if np.isfinite(pval) else 1.0,
        })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows).dropna(subset=['median_shift'])
    if df.empty:
        return df

    _, padj, _, _ = multipletests(
        df['pval_raw'].fillna(1.0), alpha=0.05, method='fdr_bh'
    )
    df['pval_adj'] = padj

    if perturb_type == 'delete':
        df['significant'] = (
            (df['pval_adj'] < 0.05) &
            (df['median_shift'].abs() > effect_threshold) &
            (df['median_shift'] < 0)
        )
    else:
        df['significant'] = (
            (df['pval_adj'] < 0.05) &
            (df['median_shift'].abs() > effect_threshold) &
            (df['median_shift'] > 0)
        )

    return df.sort_values('median_shift',
                          ascending=(perturb_type == 'delete'))


for cell_type in candidate_dfs.keys():
    print(f"\n{'='*55}")
    print(f"Stability check: {cell_type}")

    ct_isp_dir   = os.path.join(ISP_OUTPUT_DIR, cell_type.replace(' ', '_'))
    pickle_files = [
        f for f in os.listdir(ct_isp_dir)
        if f.endswith('.pickle') and 'dict_cell_embs_' in f
    ]
    if not pickle_files:
        print("   No pickle files — skipping.")
        continue

    # ── Load all shift data per gene ──────────────────────────────────────
    # Structure: gene_all_shifts[token_id] = list of per-cell shift values
    gene_all_shifts   = defaultdict(list)
    n_loaded = n_failed = 0

    for fname in pickle_files:
        fpath = os.path.join(ct_isp_dir, fname)
        try:
            with open(fpath, 'rb') as f:
                batch_dict = pickle.load(f)
            if isinstance(batch_dict, list):
                batch_dict = batch_dict[0] if batch_dict else {}

            if TARGET_LABEL not in batch_dict:
                continue

            target_d = batch_dict[TARGET_LABEL]
            for (token_id, emb_key), cos_sims in target_d.items():
                if emb_key != 'cell_emb':
                    continue
                gene_all_shifts[token_id].extend(cos_sims)
            n_loaded += 1
        except Exception as e:
            n_failed += 1

    print(f"   Loaded {n_loaded} pickle files ({n_failed} failed).")

    # ── Compute baseline ──────────────────────────────────────────────────
    ctrl_mask = ((embeddings['cell_type'] == cell_type) &
                 (embeddings['condition'] == CONTROL_LABEL))
    tgt_mask  = ((embeddings['cell_type'] == cell_type) &
                 (embeddings['condition'] == TARGET_LABEL))

    ctrl_mean = embeddings.loc[ctrl_mask, emb_feature_cols].values.mean(axis=0)
    tgt_mean  = embeddings.loc[tgt_mask,  emb_feature_cols].values.mean(axis=0)
    ctrl_norm = ctrl_mean / (np.linalg.norm(ctrl_mean) + 1e-12)
    tgt_norm  = tgt_mean  / (np.linalg.norm(tgt_mean)  + 1e-12)
    baseline_cos_sim = float(np.dot(ctrl_norm, tgt_norm))

    # Auto-detect pre-computed shifts
    sample_vals = np.array(
        list(gene_all_shifts.values())[0][:10], dtype=np.float64
    )
    sample_vals = sample_vals[np.isfinite(sample_vals)]
    if abs(np.median(sample_vals)) < 0.1:
        baseline_cos_sim = 0.0

    # Convert to shifts and filter to finite values
    gene_all_shifts_arr = {}
    for tid, vals in gene_all_shifts.items():
        arr = np.array(vals, dtype=np.float64)
        arr = arr[np.isfinite(arr)] - baseline_cos_sim
        if len(arr) >= MIN_CELLS_HALF * 2:  # need enough for splitting
            gene_all_shifts_arr[tid] = arr

    candidate_token_ids_set = {
        tk.gene_token_dict[eid]
        for eid in candidate_dfs[cell_type]['ensembl_id'].dropna()
        if eid in tk.gene_token_dict
    }

    # ── Run N_BOOTSTRAP_SPLITS random half splits ─────────────────────────
    top_hits_per_split = []

    for split_i in range(N_BOOTSTRAP_SPLITS):
        half1_shifts = {}
        half2_shifts = {}

        for tid, arr in gene_all_shifts_arr.items():
            n = len(arr)
            idx = rng_boot.permutation(n)
            split_point = int(n * SPLIT_FRACTION)
            h1 = arr[idx[:split_point]]
            h2 = arr[idx[split_point:]]
            if len(h1) >= MIN_CELLS_HALF:
                half1_shifts[tid] = h1
            if len(h2) >= MIN_CELLS_HALF:
                half2_shifts[tid] = h2

        # Separate candidate from other genes for baseline
        cand_h1 = {t: s for t, s in half1_shifts.items()
                   if t in candidate_token_ids_set}
        other_h1 = {t: s for t, s in half1_shifts.items()
                    if t not in candidate_token_ids_set}
        cand_h2 = {t: s for t, s in half2_shifts.items()
                   if t in candidate_token_ids_set}
        other_h2 = {t: s for t, s in half2_shifts.items()
                    if t not in candidate_token_ids_set}

        df1 = score_gene_shifts(cand_h1, other_h1, rng_boot,
                                perturb_type=gf_perturb_type)
        df2 = score_gene_shifts(cand_h2, other_h2, rng_boot,
                                perturb_type=gf_perturb_type)

        if df1.empty or df2.empty:
            continue

        # Top 10 by shift magnitude in each half
        top_n = 10
        top1 = set(df1.nsmallest(top_n, 'median_shift')['token_id'].tolist()
                   if gf_perturb_type == 'delete' else
                   df1.nlargest(top_n, 'median_shift')['token_id'].tolist())
        top2 = set(df2.nsmallest(top_n, 'median_shift')['token_id'].tolist()
                   if gf_perturb_type == 'delete' else
                   df2.nlargest(top_n, 'median_shift')['token_id'].tolist())

        overlap = len(top1 & top2) / top_n
        top_hits_per_split.append(overlap)

    if not top_hits_per_split:
        print("   Insufficient data for stability check.")
        continue

    mean_overlap   = np.mean(top_hits_per_split)
    std_overlap    = np.std(top_hits_per_split)
    stability_results[cell_type] = {
        'mean_top10_overlap': mean_overlap,
        'std_top10_overlap':  std_overlap,
        'n_splits':           len(top_hits_per_split),
    }

    # Interpretation
    if mean_overlap >= 0.7:
        interp = '✅ High stability — top hits consistent across splits'
    elif mean_overlap >= 0.5:
        interp = '⚠️  Moderate stability — most top hits consistent'
    else:
        interp = '❌ Low stability — top hits vary between splits'

    print(f"   Top-10 hit overlap across {len(top_hits_per_split)} splits: "
          f"{mean_overlap:.1%} ± {std_overlap:.1%}")
    print(f"   {interp}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("Stability Summary")
print('='*55)
for ct, res in stability_results.items():
    print(f"  {ct:30s}: {res['mean_top10_overlap']:.1%} "
          f"± {res['std_top10_overlap']:.1%}  "
          f"(n={res['n_splits']} splits)")

# Save
stab_df = pd.DataFrame(stability_results).T
stab_df.to_csv(os.path.join(STATS_OUTPUT_DIR, 'stability_results.csv'))
print(f"\n✅ Stability results saved.")

In [ ]:
# For each cell type, check what the stability score actually is
# and how many cells each top hit is based on
for ct, res in stability_results.items():
    print(f'\n{ct}: {res["mean_top10_overlap"]:.1%} overlap')

    # Show n_cells for top hits in full results
    ct_df = results_df[
        (results_df['cell_type'] == ct) &
        (results_df['significant'] == True)
    ].sort_values('median_cosine_shift')

    if len(ct_df) > 0:
        print(ct_df[['gene_symbol','median_cosine_shift',
                      'pval_adj','n_cells']].head(10).to_string())

In [ ]:
print(f'MIN_CELLS_PER_GENE: {MIN_CELLS_PER_GENE}')
print(f'EFFECT_SIZE_THRESHOLD: {EFFECT_SIZE_THRESHOLD}')

In [ ]:
# In your stats cell, after building results_df_ct, print the n_cells
# distribution to confirm what values are actually in the column
print(results_df_ct['n_cells'].value_counts().sort_index().head(20))
print(f'\nGenes with n_cells < {MIN_CELLS_PER_GENE}:')
print(results_df_ct[results_df_ct['n_cells'] < MIN_CELLS_PER_GENE][
    ['gene_symbol','n_cells','median_cosine_shift','pval_adj','significant']
])

## 11.Rank & Summarize Results

In [ ]:
import numpy as np, pandas as pd
from statsmodels.stats.multitest import multipletests

results_df = None

if all_results:
    results_df = pd.concat(all_results, ignore_index=True)

    # ── NaN safety ────────────────────────────────────────────────────────────
    results_df = results_df.dropna(subset=["median_cosine_shift"])
    results_df["pval_adj"]  = results_df["pval_adj"].fillna(1.0)
    results_df["pval_raw"]  = results_df["pval_raw"].fillna(1.0)

    # Ensure gene_symbol column exists
    if "gene_symbol" not in results_df.columns and "ensembl_id" in results_df.columns:
        ensembl_sym = dict(zip(
            shared_candidate_df["ensembl_id"],
            shared_candidate_df["gene_symbol"]
        ))
        results_df["gene_symbol"] = results_df["ensembl_id"].map(ensembl_sym)

    # Recompute significance uniformly from stored pval_adj + shift
    shift_col = "median_cosine_shift"
    if gf_perturb_type == "delete":
        results_df["significant"] = (
            (results_df["pval_adj"] < 0.05) &
            (results_df[shift_col].abs() > 1e-4) &
            (results_df[shift_col] < 0)
        )
    else:
        results_df["significant"] = (
            (results_df["pval_adj"] < 0.05) &
            (results_df[shift_col].abs() > 1e-4) &
            (results_df[shift_col] > 0)
        )

    results_df = results_df.sort_values(shift_col, ascending=(gf_perturb_type=="delete"))
    results_df["rank"] = range(1, len(results_df) + 1)

    out_path = os.path.join(OUTPUT_DIR, "geneformer_isp_results_full.csv")
    results_df.to_csv(out_path, index=False)
    print(f"✅ Full results saved: {out_path}")

    sig_df = results_df[results_df["significant"]]
    gene_col = "gene_symbol" if "gene_symbol" in results_df.columns else "ensembl_id"
    print(f"\n📊 {len(sig_df)} significant perturbations across all cell types.")
    for ct in passing_celltypes:
        ct_sig = sig_df[sig_df["cell_type"] == ct]
        print(f"   {ct}: {len(ct_sig)} significant genes")
        if len(ct_sig) > 0:
            print(f"     Top 5: {ct_sig.head(5)[gene_col].tolist()}")

    display(results_df[[gene_col,"cell_type","median_cosine_shift","mean_cosine_shift",
                         "median_cos_sim","pval_adj","n_cells","significant"]].head(30))
else:
    print("⚠️  No ISP results to summarize. Check earlier steps.")


## 12.Visualizations

In [ ]:
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.8,
})
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

ISP_FIG_DIR = os.path.join(OUTPUT_DIR, "isp_figures")
os.makedirs(ISP_FIG_DIR, exist_ok=True)

SIG_COLOR  = "#C0392B"
NSIG_COLOR = "#BDC3C7"

if results_df is None or results_df.empty:
    print("No results to plot.")
else:
    shift_col = "median_cosine_shift"
    padj_col  = "pval_adj"
    gene_col  = "gene_symbol" if "gene_symbol" in results_df.columns else "ensembl_id"

    for ct in passing_celltypes:
        ct_df = results_df[results_df["cell_type"] == ct].copy()
        # ── NaN safety ────────────────────────────────────────────────────
        ct_df = ct_df.dropna(subset=[shift_col])
        if ct_df.empty:
            print(f"   No valid rows for {ct} — skipping.")
            continue

        ct_df["-log10_padj"] = -np.log10(ct_df[padj_col].clip(lower=1e-300))
        n_sig = ct_df["significant"].sum()

        fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                                 gridspec_kw={"width_ratios": [2, 1.5, 1.5]})
        fig.suptitle(
            f"In-Silico Perturbation: {ct}\n"
            f"{CONTROL_LABEL} → {TARGET_LABEL}  |  "
            f"Mode: {PERTURB_MODE}-regulation  |  n={n_sig} significant hits",
            fontsize=12, fontweight="bold", y=1.02
        )

        # Panel A: Volcano
        ax = axes[0]
        colors_map = ct_df["significant"].map({True: SIG_COLOR, False: NSIG_COLOR})
        ax.scatter(ct_df[shift_col], ct_df["-log10_padj"],
                   c=colors_map, alpha=0.7, s=20, linewidths=0, rasterized=True)
        ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.6)
        ax.axhline(-np.log10(0.05), color="#7F8C8D", lw=0.8, ls=":", alpha=0.7)
        xlim = ax.get_xlim()
        ax.text(xlim[1], -np.log10(0.05) + 0.3, "FDR = 0.05",
                fontsize=7.5, ha="right", color="#7F8C8D")

        top_sig = ct_df[ct_df["significant"]].nsmallest(8, shift_col)
        for _, row in top_sig.iterrows():
            ax.annotate(str(row.get(gene_col, "")),
                        xy=(row[shift_col], row["-log10_padj"]),
                        xytext=(4, 4), textcoords="offset points", fontsize=7.5,
                        arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

        ax.set_xlabel("Median Cosine Shift (→ target state)", fontsize=11)
        ax.set_ylabel("-log₁₀(FDR-adjusted p-value)", fontsize=11)
        ax.set_title("A. Volcano Plot", fontsize=11, fontweight="bold", loc="left")
        ax.legend(handles=[
            mpatches.Patch(color=SIG_COLOR, label=f"Significant (n={n_sig})"),
            mpatches.Patch(color=NSIG_COLOR, label="Not significant"),
        ], fontsize=9, framealpha=0.8)

        # Panel B: Ranked bar chart (top 20)
        ax2 = axes[1]
        top20 = ct_df.nsmallest(20, shift_col).copy()
        bar_colors = [SIG_COLOR if s else NSIG_COLOR for s in top20["significant"]]
        labels = top20[gene_col].tolist() if gene_col in top20.columns else [str(i) for i in top20.index]
        ax2.barh(range(len(top20)), top20[shift_col].values[::-1],
                 color=bar_colors[::-1], edgecolor="none", height=0.7)
        ax2.set_yticks(range(len(top20)))
        ax2.set_yticklabels(labels[::-1], fontsize=8)
        ax2.axvline(0, color="black", lw=0.8)
        ax2.set_xlabel("Median Cosine Shift", fontsize=10)
        ax2.set_title("B. Top 20 Perturbation Hits", fontsize=11, fontweight="bold", loc="left")

        # Panel C: Effect size distribution — NaN-safe
        ax3 = axes[2]
        sig_shifts  = ct_df.loc[ct_df["significant"],  shift_col].dropna()
        nsig_shifts = ct_df.loc[~ct_df["significant"], shift_col].dropna()
        if len(sig_shifts) > 1 and sig_shifts.nunique() > 1:
            ax3.hist(sig_shifts.values, bins=min(20, len(sig_shifts)),
                     color=SIG_COLOR, alpha=0.7,
                     label=f"Significant (n={len(sig_shifts)})", density=True)
        if len(nsig_shifts) > 1 and nsig_shifts.nunique() > 1:
            ax3.hist(nsig_shifts.values, bins=min(20, len(nsig_shifts)),
                     color=NSIG_COLOR, alpha=0.5,
                     label=f"Not sig. (n={len(nsig_shifts)})", density=True)
        ax3.axvline(0, color="black", lw=0.8, ls="--", alpha=0.6)
        ax3.set_xlabel("Median Cosine Shift", fontsize=10)
        ax3.set_ylabel("Density", fontsize=10)
        ax3.set_title("C. Effect Size Distribution", fontsize=11, fontweight="bold", loc="left")
        ax3.legend(fontsize=8, framealpha=0.8)

        plt.tight_layout()
        fig_path = os.path.join(ISP_FIG_DIR, f"isp_{ct.replace(" ", "_")}.pdf")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight", format="pdf")
        plt.savefig(fig_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
        plt.show()
        print(f"   Saved: {fig_path}")

    # Multi-cell-type dot plot
    if len(passing_celltypes) >= 2:
        all_sig = results_df[results_df["significant"]].dropna(subset=[shift_col]).copy()
        if len(all_sig) > 0:
            top_genes = (all_sig.groupby(gene_col)[shift_col].mean()
                         .nsmallest(20).index.tolist())
            pivot_shift = results_df[results_df[gene_col].isin(top_genes)].pivot_table(
                index=gene_col, columns="cell_type", values=shift_col, aggfunc="mean"
            ).fillna(0)
            pivot_padj = results_df[results_df[gene_col].isin(top_genes)].pivot_table(
                index=gene_col, columns="cell_type", values=padj_col, aggfunc="mean"
            ).fillna(1.0)

            x_labels = pivot_shift.columns.tolist()
            y_labels = pivot_shift.index.tolist()
            fig_dot, ax_dot = plt.subplots(
                figsize=(max(6, len(x_labels)*2.5), max(6, len(y_labels)*0.5))
            )
            for xi, ct in enumerate(x_labels):
                for yi, gene in enumerate(y_labels):
                    sv = pivot_shift.loc[gene, ct] if ct in pivot_shift.columns else 0
                    pv = pivot_padj.loc[gene, ct]  if ct in pivot_padj.columns  else 1.0
                    sz  = max(20, -np.log10(pv + 1e-300) * 15)
                    col = SIG_COLOR if pv < 0.05 else NSIG_COLOR
                    ax_dot.scatter(xi, yi, s=sz, c=[col], alpha=0.85,
                                   linewidths=0.5, edgecolors="black")
            ax_dot.set_xticks(range(len(x_labels)))
            ax_dot.set_xticklabels(x_labels, rotation=35, ha="right", fontsize=9)
            ax_dot.set_yticks(range(len(y_labels)))
            ax_dot.set_yticklabels(y_labels, fontsize=9)
            ax_dot.set_xlabel("Cell Type", fontsize=11)
            ax_dot.set_ylabel("Gene", fontsize=11)
            ax_dot.set_title("Top Perturbation Hits Across Cell Types\n"
                             "Dot size ∝ −log₁₀(FDR p-value)  |  Red = significant",
                             fontsize=11, fontweight="bold")
            ax_dot.grid(True, alpha=0.2, linewidth=0.5)
            ax_dot.set_xlim(-0.5, len(x_labels) - 0.5)
            ax_dot.set_ylim(-0.5, len(y_labels) - 0.5)
            plt.tight_layout()
            dot_path = os.path.join(ISP_FIG_DIR, "isp_dotplot_summary.pdf")
            plt.savefig(dot_path, dpi=300, bbox_inches="tight", format="pdf")
            plt.savefig(dot_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
            plt.show()
            print(f"   Saved: {dot_path}")

print("\n✅ All ISP figures saved.")


## 13.Biological Validation — Pathway Enrichment

In [ ]:
import os, gseapy as gp

if results_df is not None and not results_df.empty:
    gene_label_col = 'gene_symbol' if 'gene_symbol' in results_df.columns else 'gene'
    for ct in passing_celltypes:
        ct_sig = results_df[(results_df['cell_type']==ct) & results_df['significant']]
        sig_genes = ct_sig[gene_label_col].dropna().unique().tolist()
        if len(sig_genes) < 5:
            print(f'⚠️  {ct}: fewer than 5 significant genes, skipping enrichment.')
            continue
        print(f'\n🔬 Pathway enrichment for {ct} ({len(sig_genes)} genes)...')
        try:
            enr = gp.enrichr(gene_list=sig_genes, gene_sets=['KEGG_2021_Human','Reactome_2022','GO_Biological_Process_2023'], organism='human', outdir=os.path.join(OUTPUT_DIR, f'enrichment_{ct.replace(" ","_")}'), cutoff=0.05, no_plot=False)
            top_paths = enr.results.nsmallest(10,'Adjusted P-value')[['Term','Adjusted P-value','Overlap','Genes']]
            print(top_paths.to_string())
            top_paths.to_csv(os.path.join(OUTPUT_DIR, f'enrichment_{ct.replace(" ","_")}_top10.csv'), index=False)
        except Exception as e:
            print(f'   ⚠️  Enrichment failed for {ct}: {e}')
else:
    print('No results available for enrichment.')

## 14.Export Final Results

In [ ]:
import os, zipfile
from google.colab import files

zip_path = '/content/geneformer_isp_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUTPUT_DIR):
        for fn in fnames:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, '/content'))

zip_path = '/content/drive/MyDrive/scFMgeneformer_isp_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUTPUT_DIR):
        for fn in fnames:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, '/content'))

print(f'✅ All results zipped: {zip_path}')
files.download(zip_path)
print('\n📥 Download triggered.')
print('\nKey output files:')
print('  geneformer_isp_results_full.csv  — full ranked gene table')
print('  candidate_genes.csv              — genes tested')
print('  isp_plot_*.png                   — volcano + bar plots per cell type')
print('  enrichment_*/                    — pathway enrichment results')
print('\n⚠️  INTERPRETATION REMINDER:')
print('  These results should be stated as:')
print('  "Geneformer predicts that perturbing gene X shifts [cell type] cell-state')
print('   representations toward [target state]."')
print('  NOT as causal claims without experimental Perturb-seq validation.')